# SBF-3: пакетная обработка JWST для произвольной пары фильтров

Ноутбук строит гладкую изофотную модель, маскирует компактные источники, измеряет SBF в двух фиксированных кольцах и считает цвет на тех же пикселях. Кольца не усредняются вслепую: одно выбирается по явно сохранённым критериям качества, второе остаётся диагностическим. Production-режим сохраняет ровно пять больших FITS; подробная покадровая диагностика остаётся в `sbf-3_debug.ipynb`.


In [ ]:
import sys
from pathlib import Path

PYTHON_EXECUTABLE = Path(sys.executable).absolute()
PROJECT_ROOT = PYTHON_EXECUTABLE.parents[2]
DATA_ROOT = PROJECT_ROOT / "data"
CODE_ROOT = PROJECT_ROOT / "code"

def resolve_project_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else PROJECT_ROOT / path

print(f"Python:       {PYTHON_EXECUTABLE}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import builtins
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
from astropy.stats import sigma_clipped_stats
from scipy.ndimage import gaussian_filter
from scipy import ndimage
from scipy.signal import fftconvolve
from astropy.convolution import Gaussian2DKernel, interpolate_replace_nans
from photutils.isophote import Ellipse, EllipseGeometry, build_ellipse_model
from photutils.segmentation import detect_sources, deblend_sources, SegmentationImage, SourceCatalog
# Полностью автономный запуск: pysiaf при импорте иначе без таймаута
# обращается к GitHub только ради проверки номера локального PRD.
import requests
_requests_get_before_stpsf = requests.get
def _offline_version_check(*args, **kwargs):
    raise requests.ConnectionError("network disabled for local SIAF version check")
requests.get = _offline_version_check
try:
    import stpsf
    import pysiaf
finally:
    requests.get = _requests_get_before_stpsf
LOCAL_SIAF_PRD_VERSION = str(pysiaf.JWST_PRD_VERSION)
from scipy.fft import fft2, fftfreq, fftshift, set_workers
from photutils.background import Background2D, MedianBackground
from astropy.stats import SigmaClip
from astropy.modeling.models import Sersic2D
from astropy.modeling.fitting import LevMarLSQFitter

_orig_print = builtins.print

def print(*args, **kwargs):
    _orig_print(f"[{time.strftime('%H:%M:%S')}]", *args, **kwargs)


# Настройки

Здесь выбираются галактика, основной фильтр SBF, фильтр цвета и пути к двум кадрам. При пакетной обработке эти значения передаются извне. Результаты сохраняются рядом с основным FITS. Численные параметры оставлены без изменений.

Ноутбук создаёт только пять проверочных FITS; численные результаты остаются в CSV.


In [ ]:
TARGET_GALAXY = globals().get("TARGET_GALAXY", "NGC 1404")
INPUT_FAMILY = "JWST_I2D_MJY_SR"
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "sbf-3.1")
DUMP_STAGE_TABLES = bool(globals().get("DUMP_STAGE_TABLES", True))
SIGNAL_FILTER = globals().get("SIGNAL_FILTER", "F356W")
COLOR_FILTER = globals().get("COLOR_FILTER", "F277W")
signal_path = resolve_project_path(globals().get("signal_path", DATA_ROOT / "NGC 1404" / "jw03055-o003_t003_nircam_clear-f356w_i2d.fits"))
color_path_value = globals().get("color_path", DATA_ROOT / "NGC 1404" / "jw03055-o003_t003_nircam_clear-f277w_i2d.fits")
color_path = None if color_path_value in (None, "") else resolve_project_path(color_path_value)

out_dir = resolve_project_path(globals().get("out_dir", signal_path.parent))
out_dir.mkdir(parents=True, exist_ok=True)
stem = signal_path.stem
DUMP_INTERMEDIATE_FITS = bool(globals().get("DUMP_INTERMEDIATE_FITS", True))


In [ ]:
TABLE_SCHEMA_VERSION = "sbf-table-2"

TABLE_METHOD_DEFAULTS = {
    ("02_center", "adopted_center"): ("center_auto_header_100px_v1", "adopted_galaxy_center"),
    ("03_mask", "central_coverage"): ("radial_mask_coverage_v1", "masked_pixel_fraction"),
    ("03_sersic", "parameters"): ("bounded_sersic_gap_fill_v1", "sersic_gap_fill_parameters"),
    ("07_isophotes", "all_fitted_points"): ("photutils_multistart_v1", "isophote_geometry"),
    ("07_isophotes", "accepted_fitted_points"): ("photutils_multistart_v1", "accepted_isophote_geometry"),
    ("07_isophotes", "full_family_nodes"): ("isophote_family_extension_v1", "model_family_nodes"),
    ("07_isophotes", "start_attempts"): ("photutils_multistart_v1", "isophote_start_attempt"),
    ("09_sources", "compact_source_catalog"): ("segmentation_aperture_photometry_v1", "compact_source_catalog"),
    ("10_psf", "contributors"): ("stpsf_hdrtab_context_v1", "effective_psf_contributor"),
    ("11_sbf", "all_measurements"): ("fft_psf_window_pr_corrected_v1", "sbf_power_and_apparent_magnitude"),
    ("11_sbf", "comparison"): ("fft_psf_window_pr_corrected_v1", "sbf_power_and_apparent_magnitude"),
    ("12_summary", "annulus_summary"): ("single_annulus_qc_selection_v1", "apparent_sbf_magnitude"),
    ("12_summary", "pipeline_variants"): ("pipeline_variant_inventory_v1", "apparent_sbf_magnitude"),
    ("13_color", "annulus_colors"): ("color_valid_overlap_of_sbf_window_median_ratio_v1", "color_index"),
    ("13_color", "color_summary"): ("selected_sbf_annulus_color_v1", "color_index"),
    ("14_measurements", "long"): ("canonical_long_table_v1", "scalar_measurement"),
}

def save_stage_table(stage, label, frame, method_id=None, quantity_family=None):
    'Сохраняет CSV с единым описанием происхождения каждой строки.'
    if not DUMP_STAGE_TABLES or frame is None:
        return None
    clean = frame.reset_index(drop=True).copy()
    default_method, default_quantity = TABLE_METHOD_DEFAULTS.get(
        (str(stage), str(label)),
        (f"sbf3_{stage}_{label}", str(label)),
    )
    primary_header = globals().get("signal_primary_header", {})
    common = {
        "schema_version": TABLE_SCHEMA_VERSION,
        "pipeline_version": PIPELINE_VERSION,
        "target_id": str(globals().get("TARGET_GALAXY", "")),
        "program_id": str(primary_header.get("PROGRAM", "")).lstrip("0") or "0",
        "observation_id": str(primary_header.get("OBSERVTN", "")),
        "signal_filter": str(globals().get("SIGNAL_FILTER", "")),
        "color_filter": str(globals().get("COLOR_FILTER", "") or "NONE"),
        "record_type": f"{stage}:{label}",
        "method_id": method_id or default_method,
        "method_variant": "default",
        "quantity_family": quantity_family or default_quantity,
        "quantity_name": quantity_family or default_quantity,
        "primary_unit": "mixed_or_not_applicable",
    }
    if "region_id" not in clean.columns:
        if "region" in clean.columns:
            clean["region_id"] = clean["region"]
        elif "region_name" in clean.columns:
            clean["region_id"] = clean["region_name"]
        else:
            clean["region_id"] = ""
    for column, value in reversed(list(common.items())):
        if column not in clean.columns:
            clean.insert(0, column, value)
        elif column in {"method_id", "method_variant", "quantity_family", "quantity_name", "primary_unit"}:
            clean[column] = clean[column].replace("", np.nan).fillna(value)
    for column in clean.select_dtypes(include=["object"]).columns:
        clean[column] = clean[column].fillna("").astype(str)
    csv_path = out_dir / f"{stem}_{stage}_{label}.csv"
    clean.to_csv(csv_path, index=False)
    print(f"[ТАБЛИЦА:{stage}] {label} -> {csv_path}")
    return csv_path


In [ ]:
ARCSEC2_PER_SR = 2.350443e-11
MJY_SR_TO_JY_PER_ARCSEC2 = 2.350443e-5
AB_ZEROPOINT_JY = 3631.0

USE_ISOPHOTE = True
DUMP_ISOPHOTES = True
USE_BKG = True
DO_DEBLEND = True
DRY = False

FIXED_CENTER = None
CENTER_GUESS_DOWN = 4
CENTER_GUESS_SMOOTH_SIGMA = 3.0
CENTER_GUESS_Q = 99.5
CENTER_GUESS_MIN_PIXELS = 50
CENTER_GUESS_VALID_FLOOR = 1e-6
CENTER_GUESS_WEIGHT_FLOOR = 1e-12
CENTER_HEADER_MAX_DELTA_PX = 100.0

BKG_BOX0 = 256
BKG_BOX2D = 256
BKG_FILTER = 5
SIGMA_STAT = 3.0
SIGMA_DET = 2.5
SIGMA_MAXIT = 5
MASK_NPIXELS = 4
SOURCE_DETECT_CONNECTIVITY = 8
DEBLEND_NLEVELS = 8
DEBLEND_CONTRAST = 0.001
DEBLEND_NPROC = 8
PREMASK_MAX_COMPACT_AREA = 35 * 35
BKG_CHECK_CORNER_FRAC = 0.10
BKG_CHECK_MIN_PIXELS = 100

SERSIC_FIT_SAMPLE_STEP = 16
SERSIC_FIT_RADIUS_PX = 2000.0
SERSIC_CENTER_BOUND_RADIUS_PX = 100.0
SERSIC_INIT_PERCENTILE = 95
SERSIC_INIT_REFF_DIV = 8.0
SERSIC_MIN_AMPLITUDE = 1e-6
SERSIC_MIN_REFF = 10.0
SERSIC_INIT_N = 4.0
SERSIC_INIT_ELLIP = 0.2
SERSIC_INIT_THETA = 0.0
SERSIC_REFF_BOUND_MIN = 5.0
SERSIC_N_BOUNDS = (0.5, 8.0)
SERSIC_ELLIP_BOUND_MAX = 0.9

HALF_SIZE = 3000
ISO_START_SMA_CANDIDATES = (40.0, 50.0, 60.0, 70.0)
ISO_START_EPS = 0.2
ISO_START_PA = 0.0
ISO_MAXSMA_FIT = 2000.0
ISO_BOOTSTRAP_MAXSMA = 200.0
ISO_MINSMA = 5.0
ISO_STEP_MAIN = 10.0
ISO_STEP_COARSE = 20.0
ISO_CHECK_SMA_RANGE = (100.0, 300.0)
ISO_FIT_USE_REAL_PIXELS = True
ISO_FIT_ALLOW_FILLED_FALLBACK = True

SBF_LIT_INNER_ARCSEC = (8.2, 16.4)
SBF_LIT_OUTER_ARCSEC = (16.4, 32.8)
SBF_MODEL_MARGIN = 1.03
MODEL_FULL_CHUNK_ROWS = 256
EXTRAP_GEOM_N = 20
EXTRAP_PROFILE_FIT_N = 20
EXTRAP_PROFILE_FIT_MIN_N = 5
MODEL_GEOM_EPS_MAX = 0.95
MODEL_FULL_EPS_CLIP_MAX = 0.90
MODEL_FULL_Q_MIN = 0.05
MODEL_FULL_BLEND_WIDTH_PX = 80.0
MODEL_FULL_BLEND_MIN_PX = 30.0
MODEL_FULL_BLEND_MAX_PX = 120.0
ISO_REAL_COVERAGE_MIN = 0.50
MODEL_SYNTH_SMA_STEP_PX = 10.0
MODEL_SYNTH_RELAX_SCALE_PX = 250.0
MODEL_CENTER_SLOPE_CLIP = 1.0e-2
MODEL_EPS_SLOPE_CLIP = 3.0e-4
MODEL_PA_SLOPE_CLIP = 3.0e-4

ROBUST_SCALE_FLOOR = 1e-12
GEOM_Q_FLOOR = 1e-3
MIN_PIXELS_ISO_CROP = 5000
MIN_PIXELS_SBF = 5000
MIN_PIXELS_SIGMA_CLIP = 100
MIN_POINTS_PK_BIN = 10
MIN_POINTS_FIT = 10
MIN_COLOR_PIXELS = 100
MIN_WORKING_ISOPHOTES = 10
MIN_ISO_CHECK_POINTS = 5
MIN_ISOPHOTES_MODEL_PROFILE = 8
MIN_ISOPHOTES_MODEL_GEOM = 5

CLIP_SIGMA_QC = 3.5
CLIP_MAXIT_QC = 5
CLIP_TAG_QC = f"{CLIP_SIGMA_QC:g}".replace(".", "p")

WIN_LEN = 8
WIN_STEP = 2
QUALITY_MAD_SCALE = 1.5
PLATEAU_MIN_LEN = 5
PLATEAU_MAX_LEN = None
DM_RANGE_MAX = 0.15
PLATEAU_ERR_WEIGHT = 0.7
PLATEAU_LEN_WEIGHT = 0.5
WINDOW_LEVEL_WEIGHT = 1.0
WINDOW_DYN_WEIGHT = 0.03
WINDOW_STD_WEIGHT = 0.03
WINDOW_MASK_WEIGHT = 0.03
PLATEAU_SPEC_MIN_SUCCESS = 3
SINGLE_WINDOW_SPEC_MIN_SUCCESS = 1

FFT_WORKERS = -1
PSF_SIZE = 129
PSF_NLAMBDA = 7
PSFREF = None
PSF_PIXEL_SCALE_RTOL = 0.05
PSF_OPD_WARN_DELTA_DAYS = 7.0
PSF_OPD_MAX_DELTA_DAYS = float(globals().get("PSF_OPD_MAX_DELTA_DAYS", 30.0))
PSF_REQUIRE_FRESH_OPD = bool(globals().get("PSF_REQUIRE_FRESH_OPD", True))
PSF_DETECTOR_OVERSAMPLE = 4
PSF_RESAMPLE_ORDER = 3
PSF_PROJECT_WSS_DIR = resolve_project_path(globals().get("PSF_PROJECT_WSS_DIR", DATA_ROOT / "wss_opd"))
STPSF_DATA_DIR = globals().get("STPSF_DATA_DIR")
FFT_E_REALIZATIONS_MAIN = 64
FFT_E_REALIZATIONS_DIAG = 64
FFT_KBINS_N = 80
FFT_K_RANGE_MAIN = (0.04, 0.25)
SBF_REGION_K_WINDOWS = [
    (0.01, 0.25),
    (0.03, 0.25),
    FFT_K_RANGE_MAIN,
]
FIT_CORR_WARN = 0.3
FIT_P1_WARN_FACTOR = 5.0
SBF_QC_MIN_USABLE_FRACTION = 0.50
SBF_QC_WARN_USABLE_FRACTION = 0.80
SBF_QC_MAX_KSHIFT_MAG = 0.15
SBF_QC_MAX_PR_OVER_P0 = 0.20
SBF_QC_MIN_CORR = 0.30

SBF_PR_ENABLE = True
SBF_PR_MAG_BIN = 0.25
SBF_PR_MLIM_OVERRIDE = None
SBF_PR_MLIM_OFFSET = 0.0
SBF_PR_FIT_WIDTH = 1.5
SBF_PR_MIN_SOURCES_REGION = 6
SBF_PR_MIN_SOURCES_GLOBAL = 20
SBF_PR_MIN_FIT_BINS = 3
SBF_PR_GAMMA_BOUNDS = (0.10, 0.70)
SBF_PR_DEFAULT_GAMMA = 0.30
SBF_PR_FALLBACK_QUANTILE = 0.80
SBF_PR_CATALOG_MIN_FLUX_JY = 0.0
SBF_PR_MAX_CORRECTION_WARN = 0.50
SBF_PR_DET_SIGMA = SIGMA_DET
SBF_PR_DET_NPIXELS = MASK_NPIXELS
SBF_PR_MAX_COMPACT_AREA = PREMASK_MAX_COMPACT_AREA
SBF_PR_DO_DEBLEND = DO_DEBLEND
SBF_PR_SOURCE_IMAGE = "img_minus_model_full_unmasked"

COLOR_CLIP_SIGMA = 3.0
COLOR_CLIP_MAXIT = 5
COLOR_GRID_TOLERANCE_PX = 0.05
COLOR_WCS_SAMPLE_CHUNK = 250000

UNROLL_N_BINS = 360
UNROLL_N_SUBANNULI = 6
UNROLL_PAD_EXTRA = 3
UNROLL_MIN_PIXELS = 10
UNROLL_SPLINE_POINTS = 1000
UNROLL_SPLINE_SMOOTH_BINS = 21

FFT_RNG_SEED = 1489


## Загрузка кадров JWST

Загружаем основной кадр для SBF и второй кадр для цвета. Фильтры читаются из заголовков FITS и сверяются с настройками. Если сетки пикселей различаются, второй кадр переносится по WCS только в нужных точках.


In [ ]:
print("loading i2d files...")

def jwst_filter_from_header(header):
    instrument = str(header.get("INSTRUME", "")).strip().upper()
    pupil = str(header.get("PUPIL", "")).strip().upper()
    filter_name = str(header.get("FILTER", "")).strip().upper()
    if instrument == "NIRCAM" and pupil.startswith("F") and pupil[-1:] in {"N", "M"}:
        return pupil
    if filter_name.startswith("F") and any(ch.isdigit() for ch in filter_name):
        return filter_name
    if pupil.startswith("F") and any(ch.isdigit() for ch in pupil):
        return pupil
    raise ValueError("cannot determine JWST filter from primary FITS header")

def checked_filter(expected, actual, role):
    if expected is None:
        return actual
    expected = str(expected).strip().upper()
    if expected != actual:
        raise ValueError(f"{role} filter mismatch: manifest={expected}, FITS={actual}")
    return expected

def checked_surface_brightness_unit(header, role):
    unit = str(header.get("BUNIT", "")).strip()
    normalized = unit.lower().replace(" ", "")
    if normalized not in {"mjy/sr", "mjysr-1"}:
        raise ValueError(f"{role} SCI BUNIT must be MJy/sr, got {unit!r}")
    return unit

def checked_jwst_product(primary_header, role):
    telescope = str(primary_header.get("TELESCOP", "")).strip().upper()
    instrument = str(primary_header.get("INSTRUME", "")).strip().upper()
    if telescope != "JWST" or instrument not in {"NIRCAM", "NIRISS"}:
        raise ValueError(f"{role} input is outside {INPUT_FAMILY}: TELESCOP={telescope!r}, INSTRUME={instrument!r}")
    return instrument

def max_same_pixel_wcs_offset(signal_wcs, color_wcs, shape):
    ny, nx = shape
    xs = np.array([0.0, nx - 1.0, 0.0, nx - 1.0, 0.5 * (nx - 1.0)])
    ys = np.array([0.0, 0.0, ny - 1.0, ny - 1.0, 0.5 * (ny - 1.0)])
    sky = signal_wcs.pixel_to_world(xs, ys)
    color_x, color_y = color_wcs.world_to_pixel(sky)
    offsets = np.hypot(color_x - xs, color_y - ys)
    return float(np.nanmax(offsets))

signal_path = Path(signal_path)
color_path = Path(color_path) if color_path is not None else None

with fits.open(signal_path, memmap=False) as hdul:
    signal_primary_header = hdul[0].header.copy()
    signal_image = hdul["SCI"].data.astype(float)
    signal_header = hdul["SCI"].header.copy()
    signal_valid = np.isfinite(signal_image)
    if "WHT" in hdul:
        try:
            signal_weight = hdul["WHT"].data
            signal_valid &= np.isfinite(signal_weight) & (signal_weight > 0)
        except Exception:
            pass

signal_instrument = checked_jwst_product(signal_primary_header, "signal")
signal_filter_actual = jwst_filter_from_header(signal_primary_header)
SIGNAL_FILTER = checked_filter(SIGNAL_FILTER, signal_filter_actual, "signal")
signal_bunit = checked_surface_brightness_unit(signal_header, "signal")
signal_wcs = WCS(signal_header).celestial
pixar_sr = float(signal_header["PIXAR_SR"])
pix_area = pixar_sr / ARCSEC2_PER_SR
print(f"[INPUT] signal={SIGNAL_FILTER} path={signal_path}")
print(f"[WCS] PIXAR_SR={pixar_sr:.8e} sr → pix_area={pix_area:.8e} arcsec^2")

color_image = None
color_header = None
color_primary_header = None
color_valid = None
color_wcs = None
color_grid_max_offset_px = np.nan
color_grid_aligned = False
color_sampling_mode = "unavailable"

if color_path is not None and color_path.exists():
    with fits.open(color_path, memmap=False) as hdul:
        color_primary_header = hdul[0].header.copy()
        color_image = hdul["SCI"].data.astype(float)
        color_header = hdul["SCI"].header.copy()
        color_valid = np.isfinite(color_image)
        if "WHT" in hdul:
            try:
                color_weight = hdul["WHT"].data
                color_valid &= np.isfinite(color_weight) & (color_weight > 0)
            except Exception:
                pass

    color_instrument = checked_jwst_product(color_primary_header, "color")
    color_filter_actual = jwst_filter_from_header(color_primary_header)
    COLOR_FILTER = checked_filter(COLOR_FILTER, color_filter_actual, "color")
    color_bunit = checked_surface_brightness_unit(color_header, "color")
    color_wcs = WCS(color_header).celestial
    color_grid_max_offset_px = max_same_pixel_wcs_offset(signal_wcs, color_wcs, signal_image.shape)
    color_grid_aligned = bool(
        color_image.shape == signal_image.shape
        and np.isfinite(color_grid_max_offset_px)
        and color_grid_max_offset_px <= COLOR_GRID_TOLERANCE_PX
    )
    color_sampling_mode = "same_grid" if color_grid_aligned else "wcs_bilinear"
    print(f"[INPUT] color={COLOR_FILTER} path={color_path}")
    print(f"[WCS] same pixel grid={color_grid_aligned}, max_offset={color_grid_max_offset_px:.4g} px")
    print(f"[WCS] color sampling mode={color_sampling_mode}")
else:
    print("[COLOR][WARN] color image unavailable; color measurement will be skipped")

output_header = signal_header.copy()
for metadata_key in ("TELESCOP", "INSTRUME", "DETECTOR", "FILTER", "PUPIL", "APERNAME", "DATE-OBS", "TIME-OBS", "PROGRAM", "OBSERVTN", "TARG_RA", "TARG_DEC"):
    metadata_value = signal_primary_header.get(metadata_key)
    if metadata_value is not None:
        output_header[metadata_key] = metadata_value
output_header["SBFVER"] = (PIPELINE_VERSION, "SBF pipeline version")
output_header["SBFFAM"] = (INPUT_FAMILY, "input data contract")
output_header["SBFTARG"] = (str(TARGET_GALAXY), "target name")
output_header["SBFSIG"] = (SIGNAL_FILTER, "SBF signal filter")
output_header["SBFCOL"] = (COLOR_FILTER or "NONE", "color filter")
output_header["SBFGRID"] = (color_sampling_mode, "color sampling mode")

print(f"{SIGNAL_FILTER} shape={signal_image.shape}, valid={signal_valid.sum()}")
if color_image is not None:
    print(f"{COLOR_FILTER} shape={color_image.shape}, valid={color_valid.sum()}")


## Начальная оценка центра

Грубо определяем центр галактики по допустимым ярким пикселям. Это лишь начальное приближение для профиля Серсика и изофот; окончательный центр SBF уточняется моделью.


In [ ]:
def guess_center_fast(img, valid, down=None, sigma=None, q=None, min_sel_pixels=None, wcs=None, log=True):
    if down is None:
        down = CENTER_GUESS_DOWN
    if sigma is None:
        sigma = CENTER_GUESS_SMOOTH_SIGMA
    if q is None:
        q = CENTER_GUESS_Q
    if min_sel_pixels is None:
        min_sel_pixels = CENTER_GUESS_MIN_PIXELS

    down = int(down)
    sigma = float(sigma)
    q = float(q)
    min_sel_pixels = int(min_sel_pixels)

    ny, nx = img.shape

    def _log_center(xc, yc, note=""):
        if not log:
            return
        msg = f"[CENTER-FAST] x={xc:.2f}, y={yc:.2f}"
        if note:
            msg += f" ({note})"
        if wcs is not None:
            ra_deg, dec_deg = wcs.pixel_to_world_values(xc, yc)
            msg += f" | RA={ra_deg:.8f} deg, Dec={dec_deg:.8f} deg"
        print(msg)

    img_d = img[::down, ::down]
    val_d = valid[::down, ::down] & np.isfinite(img_d)

    if not np.any(val_d):
        xc, yc = nx / 2.0, ny / 2.0
        _log_center(xc, yc, note="fallback: no valid downsampled pixels")
        return xc, yc

    data = np.where(val_d, img_d, 0.0).astype(np.float32)
    w = val_d.astype(np.float32)

    num = gaussian_filter(data, sigma=sigma)
    den = gaussian_filter(w, sigma=sigma)
    sm = np.divide(
        num,
        den,
        out=np.full_like(num, np.nan),
        where=den > CENTER_GUESS_VALID_FLOOR,
    )

    if not np.isfinite(sm).any():
        xc, yc = nx / 2.0, ny / 2.0
        _log_center(xc, yc, note="fallback: sm is all-NaN")
        return xc, yc

    thr = np.nanpercentile(sm, q)
    sel = np.isfinite(sm) & (sm >= thr)

    if sel.sum() < min_sel_pixels:
        y, x = np.unravel_index(np.nanargmax(sm), sm.shape)
        xc, yc = float(x * down), float(y * down)
        _log_center(xc, yc, note="fallback: argmax")
        return xc, yc

    ys, xs = np.nonzero(sel)
    ws = sm[sel] - np.nanmin(sm[sel])
    ws = np.nan_to_num(ws, nan=0.0) + CENTER_GUESS_WEIGHT_FLOOR

    x0 = float((xs * ws).sum() / ws.sum()) * down
    y0 = float((ys * ws).sum() / ws.sum()) * down
    _log_center(x0, y0, note=f"down={down}, sigma={sigma}, q={q}")
    return x0, y0


## Подготовка кадра

Вычитаем крупномасштабный фон, проверяем его остаток и строим первичную маску звёзд, шаровых скоплений и других компактных объектов.


## Вычитание фона

Сначала оцениваем постоянную составляющую фона, затем его плавное пространственное изменение. Исправленный кадр используется на всех последующих этапах.


In [ ]:
print("estimating and subtracting background...")

def subtract_pipeline_background(image, valid, role):
    image0 = np.array(image, copy=True)
    if not USE_BKG:
        print(f"[BKG:{role}] skipped")
        return image0, 0.0, np.zeros_like(image0)

    work0 = np.array(image0, copy=True)
    work0[~valid] = np.nan
    bkg0 = Background2D(
        work0,
        box_size=BKG_BOX0,
        filter_size=(BKG_FILTER, BKG_FILTER),
        sigma_clip=SigmaClip(sigma=SIGMA_STAT, maxiters=SIGMA_MAXIT),
        bkg_estimator=MedianBackground(),
        mask=~valid,
    )
    background_scalar = float(np.nanmedian(bkg0.background))
    print(f"[BKG:{role}] rough scalar background = {background_scalar:.3e}")

    image1 = image0 - background_scalar
    detection_image = np.array(image1, copy=True)
    detection_image[~valid] = np.nan
    _, detection_median, detection_std = sigma_clipped_stats(
        detection_image[np.isfinite(detection_image)],
        sigma=SIGMA_STAT,
        maxiters=SIGMA_MAXIT,
    )
    detection_threshold = detection_median + SIGMA_DET * max(detection_std, ROBUST_SCALE_FLOOR)
    background_segmentation = detect_sources(
        detection_image,
        threshold=detection_threshold,
        npixels=MASK_NPIXELS,
        connectivity=SOURCE_DETECT_CONNECTIVITY,
    )
    source_mask = np.zeros_like(image1, dtype=bool)
    if background_segmentation is not None:
        source_mask = background_segmentation.data > 0
    background_mask = (~valid) | source_mask

    background2d = Background2D(
        image1,
        box_size=BKG_BOX2D,
        filter_size=(BKG_FILTER, BKG_FILTER),
        sigma_clip=SigmaClip(sigma=SIGMA_STAT, maxiters=SIGMA_MAXIT),
        bkg_estimator=MedianBackground(),
        mask=background_mask,
    )
    background_map = np.array(background2d.background, copy=True)
    corrected = image1 - background_map
    print(
        f"[BKG:{role}] final map: "
        f"min={np.nanmin(background_map):.3e}, "
        f"med={np.nanmedian(background_map):.3e}, "
        f"max={np.nanmax(background_map):.3e}"
    )
    return corrected, background_scalar, background_map

img, bg_scalar, bg_map = subtract_pipeline_background(signal_image, signal_valid, "signal")
if color_image is not None:
    color_photometry_image, color_bg_scalar, color_bg_map = subtract_pipeline_background(color_image, color_valid, "color")
else:
    color_photometry_image = None
    color_bg_scalar = np.nan
    color_bg_map = None
output_header["SBFBGS"] = (float(bg_scalar), "signal rough background MJy/sr")
if np.isfinite(color_bg_scalar):
    output_header["SBFBGC"] = (float(color_bg_scalar), "color rough background MJy/sr")
del signal_image, bg_map
if color_image is not None:
    del color_image, color_bg_map


## Проверка вычитания фона

По углам кадра проверяем, не осталось ли грубого смещения уровня фона до трудоёмкого построения модели.


In [ ]:
print("checking residual background after subtraction...")

check = np.array(img, copy=True)
check[~signal_valid] = np.nan

ny, nx = check.shape
dx = max(1, int(nx * BKG_CHECK_CORNER_FRAC))
dy = max(1, int(ny * BKG_CHECK_CORNER_FRAC))

corners = np.concatenate([
    check[:dy, :dx].ravel(),
    check[:dy, -dx:].ravel(),
    check[-dy:, :dx].ravel(),
    check[-dy:, -dx:].ravel(),
])

corners = corners[np.isfinite(corners)]

if corners.size > BKG_CHECK_MIN_PIXELS:
    mean_c, med_c, std_c = sigma_clipped_stats(corners, sigma=SIGMA_STAT, maxiters=SIGMA_MAXIT)
    print(
        f"[BKG-CHECK] corners: "
        f"med={med_c:.3e}, mean={mean_c:.3e}, std={std_c:.3e}, N={corners.size}"
    )
else:
    print("[BKG-CHECK] too few valid corner pixels")


## Первичная маска компактных источников

Находим связанные области выше порога и оставляем среди них компактные. Максимальная площадь задаётся параметром `PREMASK_MAX_COMPACT_AREA`; сейчас он равен `35 * 35` пикселям.


In [ ]:
use_det = signal_valid & np.isfinite(img)

mean_det, med_det, std_det = sigma_clipped_stats(
    img[use_det],
    sigma=SIGMA_STAT,
    maxiters=SIGMA_MAXIT,
)

thr_det = med_det + SIGMA_DET * max(std_det, ROBUST_SCALE_FLOOR)
print(f"[ПЕРВИЧНАЯ МАСКА] med={med_det:.3e}, std={std_det:.3e}, threshold={thr_det:.3e}")

segm = detect_sources(
    img,
    threshold=thr_det,
    npixels=MASK_NPIXELS,
    connectivity=SOURCE_DETECT_CONNECTIVITY,
)
print(f"[ПЕРВИЧНАЯ МАСКА] detected {segm.nlabels if segm is not None else 0} sources")

compact_labels = []
premask_segm = segm

if segm is None:
    premask_src = np.zeros_like(img, dtype=bool)
else:
    if DO_DEBLEND:
        print("[ПЕРВИЧНАЯ МАСКА] разделение слившихся источников...")
        try:
            segm = deblend_sources(
                img,
                segm,
                npixels=MASK_NPIXELS,
                nlevels=DEBLEND_NLEVELS,
                contrast=DEBLEND_CONTRAST,
                nproc=DEBLEND_NPROC,
                progress_bar=False,
            )
        except Exception as e:
            print(f"[ПЕРВИЧНАЯ МАСКА] разделение пропущено: {e}")

    premask_segm = segm

    max_compact_area = PREMASK_MAX_COMPACT_AREA
    labels = segm.labels
    counts = np.bincount(segm.data.ravel(), minlength=labels.max() + 1)
    counts[0] = 0

    compact_labels = [int(lab) for lab in labels if 0 < counts[lab] <= max_compact_area]

    compact_lookup = np.zeros(counts.size, dtype=bool)
    compact_lookup[compact_labels] = True
    premask_src = compact_lookup[segm.data]

premask_compact_labels = np.array(compact_labels, dtype=int)

premask = (~signal_valid) | premask_src

print(f"[ПЕРВИЧНАЯ МАСКА] оставлено компактных областей = {len(premask_compact_labels)}")
print(f"[ПЕРВИЧНАЯ МАСКА] доля закрытых пикселей = {100.0 * premask.sum() / premask.size:.2f}%")


## Принятый центр галактики

Берём заданный вручную центр либо сравниваем автоматическую оценку с `TARG_RA/TARG_DEC` из заголовка FITS. При расхождении больше 100 пикселей используем координаты цели из заголовка.


In [ ]:
print("choosing galaxy center...")

auto_x, auto_y = np.nan, np.nan
header_x, header_y = np.nan, np.nan
header_ra = signal_primary_header.get("TARG_RA")
header_dec = signal_primary_header.get("TARG_DEC")
auto_header_delta_px = np.nan

if FIXED_CENTER is None:
    auto_x, auto_y = guess_center_fast(
        img,
        signal_valid & (~premask),
        down=CENTER_GUESS_DOWN,
        sigma=CENTER_GUESS_SMOOTH_SIGMA,
        q=CENTER_GUESS_Q,
        min_sel_pixels=CENTER_GUESS_MIN_PIXELS,
        wcs=signal_wcs,
        log=True,
    )
    header_center_valid = False
    if header_ra is not None and header_dec is not None:
        try:
            header_ra = float(header_ra)
            header_dec = float(header_dec)
            header_x, header_y = signal_wcs.world_to_pixel_values(header_ra, header_dec)
            header_x, header_y = float(header_x), float(header_y)
            header_center_valid = bool(
                np.isfinite(header_x) and np.isfinite(header_y)
                and 0.0 <= header_x < img.shape[1]
                and 0.0 <= header_y < img.shape[0]
            )
        except Exception as exc:
            print(f"[CENTER-HEADER][WARN] cannot transform TARG_RA/TARG_DEC: {exc}")
            header_ra, header_dec = np.nan, np.nan

    if header_center_valid:
        auto_header_delta_px = float(np.hypot(auto_x - header_x, auto_y - header_y))
        print(
            f"[CENTER-HEADER] x={header_x:.2f}, y={header_y:.2f} | "
            f"RA={header_ra:.8f} deg, Dec={header_dec:.8f} deg | "
            f"auto-header delta={auto_header_delta_px:.2f} px"
        )
        if auto_header_delta_px > CENTER_HEADER_MAX_DELTA_PX:
            x0_center, y0_center = header_x, header_y
            center_src = "fits-target"
            print(
                f"[CENTER-CHECK] delta>{CENTER_HEADER_MAX_DELTA_PX:.1f} px; "
                "using TARG_RA/TARG_DEC"
            )
        else:
            x0_center, y0_center = auto_x, auto_y
            center_src = "auto-fast"
            print(
                f"[CENTER-CHECK] delta<={CENTER_HEADER_MAX_DELTA_PX:.1f} px; "
                "using automatic center"
            )
    else:
        x0_center, y0_center = auto_x, auto_y
        center_src = "auto-fast-no-valid-header"
        print("[CENTER-HEADER][WARN] valid in-frame TARG_RA/TARG_DEC unavailable; using automatic center")
else:
    x0_center, y0_center = FIXED_CENTER
    ra_deg, dec_deg = signal_wcs.pixel_to_world_values(x0_center, y0_center)
    print(f"[CENTER-FIXED] x={x0_center:.2f}, y={y0_center:.2f} | RA={ra_deg:.8f} deg, Dec={dec_deg:.8f} deg")
    center_src = "fixed"

print(f"[CENTER] using ({x0_center:.2f}, {y0_center:.2f}) [{center_src}]")
center_ra_deg, center_dec_deg = signal_wcs.pixel_to_world_values(x0_center, y0_center)
output_header["SBFCENS"] = (center_src, "adopted center source")
if np.isfinite(auto_header_delta_px):
    output_header["SBFCDEL"] = (float(auto_header_delta_px), "auto-header center delta, pix")
center_df = pd.DataFrame([{
    "target": TARGET_GALAXY, "source": center_src,
    "x_pixel": float(x0_center), "y_pixel": float(y0_center),
    "ra_deg": float(center_ra_deg), "dec_deg": float(center_dec_deg),
    "auto_x_pixel": float(auto_x), "auto_y_pixel": float(auto_y),
    "header_x_pixel": float(header_x), "header_y_pixel": float(header_y),
    "header_ra_deg": float(header_ra) if header_ra is not None else np.nan,
    "header_dec_deg": float(header_dec) if header_dec is not None else np.nan,
    "auto_header_delta_px": float(auto_header_delta_px),
    "decision_threshold_px": float(CENTER_HEADER_MAX_DELTA_PX),
}])
save_stage_table("02_center", "adopted_center", center_df)


## Запасная модель Серсика

Строим простую гладкую модель по замаскированному кадру. Параметры оцениваются в окрестности выбранного центра, но модель подставляется во все NaN и пиксели с WHT≤0 на всём кадре, включая щели между детекторами. Это вспомогательное заполнение для поиска изофот, а не итоговая модель SBF.


In [ ]:
fit_radius = float(SERSIC_FIT_RADIUS_PX)
fit_x1 = max(0, int(np.floor(x0_center - fit_radius)))
fit_x2 = min(img.shape[1], int(np.ceil(x0_center + fit_radius + 1)))
fit_y1 = max(0, int(np.floor(y0_center - fit_radius)))
fit_y2 = min(img.shape[0], int(np.ceil(y0_center + fit_radius + 1)))
img_fit_local = img[fit_y1:fit_y2, fit_x1:fit_x2]
mask_fit_local = premask[fit_y1:fit_y2, fit_x1:fit_x2]
local_y, local_x = np.indices(img_fit_local.shape, dtype=np.float32)
local_r2 = (local_x + fit_x1 - x0_center) ** 2 + (local_y + fit_y1 - y0_center) ** 2
use_fit_local = (~mask_fit_local) & np.isfinite(img_fit_local) & (local_r2 <= fit_radius ** 2)

sample_step = int(SERSIC_FIT_SAMPLE_STEP)
y_fit = (local_y[use_fit_local] + fit_y1)[::sample_step]
x_fit = (local_x[use_fit_local] + fit_x1)[::sample_step]
z_fit = img_fit_local[use_fit_local][::sample_step]
del local_y, local_x, local_r2

print(f"[SERSIC] fit sample size = {z_fit.size}")

if z_fit.size < 100:
    raise RuntimeError(f"[SERSIC] too few local pixels for auxiliary fill model: {z_fit.size}")
amp0 = float(np.nanpercentile(z_fit, SERSIC_INIT_PERCENTILE))

r_eff0 = float(fit_radius / SERSIC_INIT_REFF_DIV)

sersic_init = Sersic2D(

    amplitude=max(amp0, SERSIC_MIN_AMPLITUDE),

    r_eff=max(r_eff0, SERSIC_MIN_REFF),

    n=SERSIC_INIT_N,

    x_0=float(x0_center),
    y_0=float(y0_center),

    ellip=SERSIC_INIT_ELLIP,

    theta=SERSIC_INIT_THETA,
)

sersic_init.amplitude.bounds = (SERSIC_MIN_AMPLITUDE, None)

sersic_init.r_eff.bounds = (SERSIC_REFF_BOUND_MIN, float(max(img.shape)))

sersic_init.n.bounds = SERSIC_N_BOUNDS

center_bound = float(SERSIC_CENTER_BOUND_RADIUS_PX)
sersic_init.x_0.bounds = (max(0.0, x0_center - center_bound), min(float(img.shape[1] - 1), x0_center + center_bound))
sersic_init.y_0.bounds = (max(0.0, y0_center - center_bound), min(float(img.shape[0] - 1), y0_center + center_bound))

sersic_init.ellip.bounds = (0.0, SERSIC_ELLIP_BOUND_MAX)

sersic_init.theta.bounds = (-np.pi / 2.0, np.pi / 2.0)

fitter = LevMarLSQFitter()

try:
    sersic_candidate = fitter(sersic_init, x_fit, y_fit, z_fit)
    candidate_values = np.array([
        sersic_candidate.amplitude.value, sersic_candidate.r_eff.value,
        sersic_candidate.n.value, sersic_candidate.x_0.value,
        sersic_candidate.y_0.value, sersic_candidate.ellip.value,
        sersic_candidate.theta.value,
    ], dtype=float)
    if not np.all(np.isfinite(candidate_values)):
        raise ValueError("non-finite fitted parameters")
    if float(sersic_candidate.amplitude.value) <= SERSIC_MIN_AMPLITUDE:
        raise ValueError("fitted amplitude collapsed to zero")
    if np.hypot(sersic_candidate.x_0.value - x0_center, sersic_candidate.y_0.value - y0_center) > center_bound:
        raise ValueError("fitted center left the auxiliary-model bounds")
    sersic_fit = sersic_candidate
    print(
        "[SERSIC] fitted:",
        f"amp={float(sersic_fit.amplitude.value):.3e},",
        f"r_eff={float(sersic_fit.r_eff.value):.2f},",
        f"n={float(sersic_fit.n.value):.2f},",
        f"x0={float(sersic_fit.x_0.value):.2f},",
        f"y0={float(sersic_fit.y_0.value):.2f},",
        f"ellip={float(sersic_fit.ellip.value):.3f},",
        f"theta={float(sersic_fit.theta.value):.3f}",
    )
except Exception as e:
    print(f"[SERSIC] fit failed, fallback to initial model: {e}")
    sersic_fit = sersic_init

sm = np.array(img, copy=True)

bad = (~signal_valid) | (~np.isfinite(sm))

bad_y, bad_x = np.nonzero(bad)
fill_values = np.asarray(sersic_fit(bad_x, bad_y), dtype=float)
if not np.all(np.isfinite(fill_values)) or np.any(fill_values < 0.0):
    print("[SERSIC][WARN] invalid auxiliary fill values; using initial positive model")
    sersic_fit = sersic_init
    fill_values = np.asarray(sersic_fit(bad_x, bad_y), dtype=float)
if not np.all(np.isfinite(fill_values)):
    raise RuntimeError("[SERSIC] auxiliary model could not provide finite hole filling")
sm[bad] = fill_values
remaining_bad = int(np.count_nonzero(bad & (~np.isfinite(sm))))
if remaining_bad:
    raise RuntimeError(f"[SERSIC] {remaining_bad} detector-gap pixels remain non-finite after filling")
print(f"[SERSIC] global detector-gap fill: {bad.sum()} invalid pixels replaced; remaining non-finite={remaining_bad}")
sersic_parameters_df = pd.DataFrame([{
    "amplitude": float(sersic_fit.amplitude.value),
    "r_eff": float(sersic_fit.r_eff.value), "n": float(sersic_fit.n.value),
    "x0": float(sersic_fit.x_0.value), "y0": float(sersic_fit.y_0.value),
    "ellip": float(sersic_fit.ellip.value), "theta_rad": float(sersic_fit.theta.value),
    "fit_radius_px": float(fit_radius), "purpose": "global NaN/WHT gap fill for isophote solver only",
}])
save_stage_table("03_sersic", "parameters", sersic_parameters_df)


## Вырезка вокруг галактики

Ограничиваем расчёт областью вокруг галактики, чтобы ускорить и стабилизировать поиск изофот. Координаты сохраняются для переноса результата обратно на полный кадр.


In [ ]:
print("building cutout...")

ny, nx = sm.shape
x1 = max(0, int(x0_center - HALF_SIZE))
x2 = min(nx, int(x0_center + HALF_SIZE))
y1 = max(0, int(y0_center - HALF_SIZE))
y2 = min(ny, int(y0_center + HALF_SIZE))

img_real_c = np.asarray(img[y1:y2, x1:x2], dtype=float)
img_fill_c = np.asarray(sm[y1:y2, x1:x2], dtype=float)
img_c = np.array(img_real_c, copy=True)
valid_c = signal_valid[y1:y2, x1:x2]
mask_c = premask_src[y1:y2, x1:x2]

x0c = x0_center - x1
y0c = y0_center - y1

print(f"[CUTOUT] bounds = x[{x1}:{x2}], y[{y1}:{y2}]")
print(f"[CUTOUT] center in crop = ({x0c:.2f}, {y0c:.2f})")
print(f"[CUTOUT] shape = {img_c.shape}")
print(f"[CUTOUT] real finite = {int(np.isfinite(img_real_c).sum())}, filled finite = {int(np.isfinite(img_fill_c).sum())}")
cutout_header = output_header.copy()
if "CRPIX1" in cutout_header:
    cutout_header["CRPIX1"] = float(cutout_header["CRPIX1"]) - x1
if "CRPIX2" in cutout_header:
    cutout_header["CRPIX2"] = float(cutout_header["CRPIX2"]) - y1


## Данные для поиска изофот

Основной вариант использует только реальные пригодные пиксели. Заполненный кадр включается лишь как запасной вариант, если алгоритму не хватает конечных значений.


In [ ]:
print("preparing data for isophote fit...")

data_real = np.asarray(img_real_c, dtype=float).copy()
data_real[(~valid_c) | mask_c] = np.nan
ok_real = np.isfinite(data_real)
data_real_ma = np.ma.array(data_real, mask=~ok_real)

data_fill = np.asarray(img_fill_c, dtype=float).copy()
data_fill[mask_c] = np.nan
ok_fill = np.isfinite(data_fill)
data_fill_ma = np.ma.array(data_fill, mask=~ok_fill)

iso_fit_primary_mode = "real_only" if ISO_FIT_USE_REAL_PIXELS else "filled_primary"
iso_fit_fallback_enabled = bool(ISO_FIT_ALLOW_FILLED_FALLBACK)

print(f"[ISO] real finite pixels in crop = {int(ok_real.sum())}")
print(f"[ISO] filled finite pixels in crop = {int(ok_fill.sum())}")
print(f"[ISO] masked by source mask = {mask_c.sum()}, old invalid in crop = {(~valid_c).sum()}")
print(f"[ISO] primary fit mode = {iso_fit_primary_mode}, filled fallback enabled = {iso_fit_fallback_enabled}")

if ISO_FIT_USE_REAL_PIXELS and ok_real.sum() < MIN_PIXELS_ISO_CROP and not ISO_FIT_ALLOW_FILLED_FALLBACK:
    raise RuntimeError(f"[ISO] too few valid real-data pixels in crop for real-only fit: N={int(ok_real.sum())}")
if (not ISO_FIT_USE_REAL_PIXELS) and ok_fill.sum() < MIN_PIXELS_ISO_CROP:
    raise RuntimeError(f"[ISO] too few valid filled pixels in crop for isophote fit: N={int(ok_fill.sum())}")


## Поиск изофот

Перебираем стартовые большие полуоси 40, 50, 60 и 70 пикселей. Сначала центр, эллиптичность и позиционный угол свободны. Фиксированный центр используется только как последний запасной вариант; угол и эллиптичность остаются свободными. Все попытки записываются в CSV.


In [ ]:
print("fitting isophotes with multiple starting radii...")

minsma = ISO_MINSMA
fit_maxsma = ISO_MAXSMA_FIT
bootstrap_maxsma = min(ISO_BOOTSTRAP_MAXSMA, fit_maxsma)
fit_attempt_rows = []
fit_attempts_path = out_dir / f"{stem}_07_isophotes_start_attempts.csv"

def _isolist_stats(isolist_local):
    if isolist_local is None or len(isolist_local) == 0:
        return {
            "n_isophotes": 0, "max_sma_px": np.nan, "center_delta_px": np.nan,
            "x0_median": np.nan, "y0_median": np.nan, "eps_median": np.nan,
            "pa_median_rad": np.nan, "last_stop_code": np.nan, "stop_codes": "",
        }
    sma_values = np.asarray([iso.sma for iso in isolist_local], dtype=float)
    x_values = np.asarray([iso.x0 for iso in isolist_local], dtype=float)
    y_values = np.asarray([iso.y0 for iso in isolist_local], dtype=float)
    eps_values = np.asarray([iso.eps for iso in isolist_local], dtype=float)
    pa_values = np.asarray([iso.pa for iso in isolist_local], dtype=float)
    stop_codes = [getattr(iso, "stop_code", None) for iso in isolist_local]
    x_median = float(np.nanmedian(x_values))
    y_median = float(np.nanmedian(y_values))
    return {
        "n_isophotes": int(len(isolist_local)),
        "max_sma_px": float(np.nanmax(sma_values)),
        "center_delta_px": float(np.hypot(x_median - x0c, y_median - y0c)),
        "x0_median": x_median, "y0_median": y_median,
        "eps_median": float(np.nanmedian(eps_values)),
        "pa_median_rad": float(np.nanmedian(pa_values)),
        "last_stop_code": stop_codes[-1] if stop_codes else np.nan,
        "stop_codes": ";".join(str(code) for code in sorted(set(stop_codes), key=str)),
    }

def _working_isolist(isolist_local, stats_local=None):
    stats_local = _isolist_stats(isolist_local) if stats_local is None else stats_local
    return (
        stats_local["n_isophotes"] >= MIN_WORKING_ISOPHOTES
        and np.isfinite(stats_local["max_sma_px"])
        and np.isfinite(stats_local["center_delta_px"])
        and stats_local["center_delta_px"] <= CENTER_HEADER_MAX_DELTA_PX
    )

def _save_fit_attempts():
    save_stage_table("07_isophotes", "start_attempts", pd.DataFrame(fit_attempt_rows))

def _fit_once(data_ma_try, mode_label, n_valid_try, phase, start_sma, maxsma_try, step_try, fix_center_try):
    geom_try = EllipseGeometry(
        x0=x0c, y0=y0c, sma=float(start_sma), eps=ISO_START_EPS, pa=ISO_START_PA,
    )
    isolist_local = None
    error_local = None
    try:
        isolist_local = Ellipse(data_ma_try, geom_try).fit_image(
            minsma=minsma, maxsma=maxsma_try, step=step_try, linear=True,
            fix_center=fix_center_try, fix_pa=False, fix_eps=False,
        )
    except Exception as exc:
        error_local = f"{type(exc).__name__}: {exc}"
    stats_local = _isolist_stats(isolist_local)
    usable_local = _working_isolist(isolist_local, stats_local)
    status_local = "working" if usable_local else ("error" if error_local else "too_short_or_shifted")
    fit_attempt_rows.append({
        "phase": phase, "dataset": mode_label, "n_finite": int(n_valid_try),
        "start_sma_px": float(start_sma), "minsma_px": float(minsma),
        "maxsma_requested_px": float(maxsma_try), "step_px": float(step_try),
        "linear": True, "fix_center": bool(fix_center_try),
        "fix_pa": False, "fix_eps": False, "status": status_local,
        "error": error_local or "", **stats_local,
    })
    _save_fit_attempts()
    print(
        f"[ISO:{phase}] dataset={mode_label}, start={start_sma:g}, step={step_try:g}, "
        f"fix_center={fix_center_try}: N={stats_local['n_isophotes']}, "
        f"maxsma={stats_local['max_sma_px']:.1f}, center_delta={stats_local['center_delta_px']:.1f}, "
        f"status={status_local}{'; ' + error_local if error_local else ''}"
    )
    return isolist_local, stats_local, error_local, usable_local

def _fit_dataset_multistart(data_ma_try, mode_label, n_valid_try, fix_center_try):
    bootstrap_candidates = []
    for start_sma in ISO_START_SMA_CANDIDATES:
        best_for_start = None
        for step_try in (ISO_STEP_MAIN, ISO_STEP_COARSE):
            result = _fit_once(
                data_ma_try, mode_label, n_valid_try, "bootstrap", start_sma,
                bootstrap_maxsma, step_try, fix_center_try,
            )
            isolist_try, stats_try, error_try, usable_try = result
            if usable_try:
                best_for_start = {
                    "start_sma": float(start_sma), "step": float(step_try),
                    "stats": stats_try, "error": error_try,
                }
                break
        if best_for_start is not None:
            bootstrap_candidates.append(best_for_start)

    bootstrap_candidates.sort(
        key=lambda candidate: (
            candidate["stats"]["max_sma_px"],
            candidate["stats"]["n_isophotes"],
            -candidate["stats"]["center_delta_px"],
        ),
        reverse=True,
    )
    for candidate in bootstrap_candidates:
        start_sma = candidate["start_sma"]
        preferred_step = candidate["step"]
        full_steps = (preferred_step,) if preferred_step == ISO_STEP_COARSE else (ISO_STEP_MAIN, ISO_STEP_COARSE)
        for step_try in full_steps:
            result = _fit_once(
                data_ma_try, mode_label, n_valid_try, "full", start_sma,
                fit_maxsma, step_try, fix_center_try,
            )
            isolist_try, stats_try, error_try, usable_try = result
            if usable_try:
                signature = (
                    f"start_sma={start_sma:g}, step={step_try:g}, linear=True, "
                    f"fix_center={fix_center_try}, fix_pa=False, fix_eps=False"
                )
                return isolist_try, signature, error_try, start_sma, step_try
    return None, None, None, None, None

fit_datasets = []
if ISO_FIT_USE_REAL_PIXELS:
    fit_datasets.append(("real_only", data_real_ma, int(ok_real.sum())))
    if ISO_FIT_ALLOW_FILLED_FALLBACK:
        fit_datasets.append(("filled_fallback", data_fill_ma, int(ok_fill.sum())))
else:
    fit_datasets.append(("filled_primary", data_fill_ma, int(ok_fill.sum())))

isolist = None
last_err = None
iso_fit_mode_used = None
iso_fit_signature_used = None
iso_start_sma_used = None
iso_step_used = None
iso_fix_center_used = None
fit_attempt_summaries = []

for mode_label, data_ma_try, n_valid_try in fit_datasets:
    print(f"[ISO] trying dataset={mode_label}, N_finite={n_valid_try}")
    if n_valid_try < MIN_PIXELS_ISO_CROP:
        msg = f"dataset={mode_label}: too few finite pixels N={n_valid_try}"
        print(f"[ISO] skipping {msg}")
        fit_attempt_summaries.append(msg)
        continue

    for fix_center_try in (False, True):
        if fix_center_try:
            print(f"[ISO] free-center starts failed for {mode_label}; retrying with fixed center")
        result = _fit_dataset_multistart(data_ma_try, mode_label, n_valid_try, fix_center_try)
        isolist_try, signature_try, last_err_try, start_sma_try, step_try = result
        if isolist_try is not None and len(isolist_try) >= MIN_WORKING_ISOPHOTES:
            isolist = isolist_try
            last_err = last_err_try
            iso_fit_mode_used = mode_label
            iso_fit_signature_used = signature_try
            iso_start_sma_used = start_sma_try
            iso_step_used = step_try
            iso_fix_center_used = fix_center_try
            break
        fit_attempt_summaries.append(
            f"dataset={mode_label}, fix_center={fix_center_try}: no working start"
        )
    if isolist is not None:
        break

if isolist is None or len(isolist) < MIN_WORKING_ISOPHOTES:
    summary_text = " | ".join(fit_attempt_summaries) if fit_attempt_summaries else "no fit attempts recorded"
    raise RuntimeError(
        f"[ISO] isolist too short after all starts and datasets; last_err={last_err}; "
        f"attempts={summary_text}; details={fit_attempts_path}"
    )

print(f"[ISO] fit_image maxsma = {fit_maxsma}")
print(f"[ISO] fit dataset used = {iso_fit_mode_used}, solver path = {iso_fit_signature_used}")
print(f"[ISO] isolist N={len(isolist)}, maxsma≈{float(np.nanmax([iso.sma for iso in isolist])):.1f} px")
print(f"[ISO] all start attempts -> {fit_attempts_path}")

x0_fit = float(np.nanmedian([iso.x0 for iso in isolist]))
y0_fit = float(np.nanmedian([iso.y0 for iso in isolist]))
eps_fit = float(np.nanmedian([iso.eps for iso in isolist]))
pa_fit = float(np.nanmedian([iso.pa for iso in isolist]))
isophote_center_delta_px = float(np.hypot(x0_fit - x0c, y0_fit - y0c))
print(f"[ISO] fitted center≈({x0_fit:.1f}, {y0_fit:.1f}) in crop, eps≈{eps_fit:.3f}, pa≈{pa_fit:.3f} rad")
print(f"[ISO] fitted-adopted center delta={isophote_center_delta_px:.2f} px")
if isophote_center_delta_px > CENTER_HEADER_MAX_DELTA_PX:
    raise RuntimeError(
        f"[ISO] fitted center moved {isophote_center_delta_px:.1f} px from adopted target center; "
        "refusing to measure SBF around a different object"
    )
output_header["SBFIDEL"] = (isophote_center_delta_px, "isophote-adopted center delta, pix")
output_header["SBFISMA"] = (float(iso_start_sma_used), "selected isophote start sma, pix")
output_header["SBFIFIX"] = (bool(iso_fix_center_used), "isophote center fixed during fit")


## Изофотная модель на вырезке

Строим двумерную модель по найденным изофотам и проверяем остатки на вырезке до переноса на полный кадр.


In [ ]:
print("building isophote model...")

model_c = build_ellipse_model(img_real_c.shape, isolist)
model_c = np.where(np.isfinite(model_c) & (model_c > 0.0), model_c, np.nan)

model = np.full_like(img, np.nan)

resid = np.full_like(img, np.nan)

model[y1:y2, x1:x2] = model_c

resid[y1:y2, x1:x2] = img_real_c - model_c

resid[premask] = np.nan

finite_resid = np.isfinite(resid)
print(
    f"[CHK] resid: finite={finite_resid.sum()}, "
    f"min={np.nanmin(resid):.3e}, med={np.nanmedian(resid):.3e}, max={np.nanmax(resid):.3e}"
)

finite_model_c = np.isfinite(model_c)
print(
    f"[CHK] model_c: finite={finite_model_c.sum()}, "
    f"min={np.nanmin(model_c):.3e}, med={np.nanmedian(model_c):.3e}, max={np.nanmax(model_c):.3e}"
)


## Изофотная модель полного кадра

Переносим найденные изофоты на полный кадр и плавно продолжаем их до внешней границы второго кольца. Полученный массив `model_full` является итоговой гладкой моделью галактики.


In [ ]:
from photutils.isophote import IsophoteList

print("building continuous 2D full-frame isophotal model for Jensen/TRGB-SBF annuli...")

pix_scale = float(np.sqrt(pix_area))
r1_in, r1_out = SBF_LIT_INNER_ARCSEC[0] / pix_scale, SBF_LIT_INNER_ARCSEC[1] / pix_scale
r2_in, r2_out = SBF_LIT_OUTER_ARCSEC[0] / pix_scale, SBF_LIT_OUTER_ARCSEC[1] / pix_scale
lit_outer_r_px = float(r2_out)

class _SyntheticIso:
    __slots__ = ("sma", "intens", "eps", "pa", "x0", "y0", "grad", "a3", "b3", "a4", "b4")

    def __init__(self, sma, intens, eps, pa, x0, y0, grad):
        self.sma = float(sma)
        self.intens = float(intens)
        self.eps = float(eps)
        self.pa = float(pa)
        self.x0 = float(x0)
        self.y0 = float(y0)
        self.grad = float(grad)
        self.a3 = 0.0
        self.b3 = 0.0
        self.a4 = 0.0
        self.b4 = 0.0

def _clip_slope(value, limit):
    try:
        value = float(value)
    except Exception:
        return 0.0
    if not np.isfinite(value):
        return 0.0
    return float(np.clip(value, -abs(limit), abs(limit)))

def _weighted_line_fit(x, y, w):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    good = np.isfinite(x) & np.isfinite(y) & np.isfinite(w) & (w > 0.0)
    x = x[good]
    y = y[good]
    w = w[good]
    if x.size == 0:
        return np.nan, 0.0
    if x.size == 1:
        return float(y[0]), 0.0
    try:
        slope, intercept = np.polyfit(x, y, 1, w=w)
    except Exception:
        slope, intercept = np.polyfit(x, y, 1)
    return float(intercept), float(slope)

def _saturating_extension(delta, slope, scale):
    delta = np.asarray(delta, dtype=float)
    if scale <= 0.0:
        return slope * delta
    return slope * scale * (1.0 - np.exp(-delta / scale))

def _gradient_from_profile(sma, intens):
    sma = np.asarray(sma, dtype=float)
    intens = np.asarray(intens, dtype=float)
    if sma.size == 1:
        return np.array([-abs(intens[0]) / max(sma[0], 1.0)], dtype=float)
    grad = np.gradient(intens, sma, edge_order=1)
    bad = (~np.isfinite(grad)) | (grad >= 0.0)
    if np.any(bad):
        safe = np.zeros_like(grad)
        safe[:] = -np.maximum(np.abs(intens), 1e-6) / np.maximum(sma, 1.0)
        grad[bad] = safe[bad]
    return grad

iso_rows = []
for iso in isolist:
    sma_i = float(iso.sma)
    intens_i = float(getattr(iso, "intens", np.nan))
    x0_i = float(iso.x0)
    y0_i = float(iso.y0)
    eps_i = float(iso.eps)
    pa_i = float(iso.pa)
    grad_i = float(getattr(iso, "grad", np.nan))

    real_data_frac = np.nan
    sampled_real = 0
    sampled_total = 0
    try:
        xs, ys = iso.sampled_coordinates()
        xi = np.rint(xs).astype(int)
        yi = np.rint(ys).astype(int)
        inside = (xi >= 0) & (xi < valid_c.shape[1]) & (yi >= 0) & (yi < valid_c.shape[0])
        sampled_total = int(np.count_nonzero(inside))
        if sampled_total > 0:
            sampled_real = int(np.count_nonzero(valid_c[yi[inside], xi[inside]]))
            real_data_frac = float(sampled_real / sampled_total)
    except Exception:
        pass

    iso_rows.append({
        "sma": sma_i,
        "intens": intens_i,
        "x0_crop": x0_i,
        "y0_crop": y0_i,
        "x0_full": float(x1 + x0_i),
        "y0_full": float(y1 + y0_i),
        "eps": eps_i,
        "pa": pa_i,
        "grad": grad_i,
        "real_data_frac": real_data_frac,
        "sampled_real": sampled_real,
        "sampled_total": sampled_total,
    })

iso_df = pd.DataFrame(iso_rows).sort_values("sma").reset_index(drop=True)
iso_df["pa_unwrap"] = np.unwrap(iso_df["pa"].to_numpy(dtype=float))

fit_good = (
    np.isfinite(iso_df["sma"])
    & np.isfinite(iso_df["intens"])
    & np.isfinite(iso_df["x0_full"])
    & np.isfinite(iso_df["y0_full"])
    & np.isfinite(iso_df["eps"])
    & np.isfinite(iso_df["pa_unwrap"])
    & (iso_df["sma"] > 0.0)
    & (iso_df["intens"] > 0.0)
    & (iso_df["eps"] >= 0.0)
    & (iso_df["eps"] < MODEL_GEOM_EPS_MAX)
)
fit_df = iso_df.loc[fit_good].copy().reset_index(drop=True)
if len(fit_df) < MIN_ISOPHOTES_MODEL_PROFILE:
    raise RuntimeError(f"[МОДЕЛЬ ПОЛНОГО КАДРА] too few valid fitted isophotes for 2D family: N={len(fit_df)}")

fit_df = fit_df.drop_duplicates(subset="sma", keep="last").sort_values("sma").reset_index(drop=True)

sma_profile = fit_df["sma"].to_numpy(dtype=float)
int_profile = fit_df["intens"].to_numpy(dtype=float)
fitted_sma_min_px = float(np.nanmin(sma_profile))
fitted_sma_max_px = float(np.nanmax(sma_profile))

real_cov = fit_df["real_data_frac"].to_numpy(dtype=float)
real_cov_use = np.isfinite(real_cov) & (real_cov >= ISO_REAL_COVERAGE_MIN)
if np.any(real_cov_use):
    trusted_sma_max_px = float(np.nanmax(fit_df.loc[real_cov_use, "sma"]))
else:
    trusted_sma_max_px = fitted_sma_max_px

x0_model_full = float(np.nanmedian(fit_df["x0_full"]))
y0_model_full = float(np.nanmedian(fit_df["y0_full"]))

edge_row = fit_df.iloc[-1]
sma_edge = float(edge_row["sma"])
I_at_fit_edge = float(edge_row["intens"])
x0_edge = float(edge_row["x0_full"])
y0_edge = float(edge_row["y0_full"])
eps_edge = float(np.clip(edge_row["eps"], 0.0, MODEL_FULL_EPS_CLIP_MAX))
pa_edge = float(edge_row["pa_unwrap"])

fit_tail_n = min(EXTRAP_GEOM_N, len(fit_df))
tail_df = fit_df.tail(fit_tail_n).copy()
tail_w = np.clip(np.nan_to_num(tail_df["real_data_frac"].to_numpy(dtype=float), nan=0.25), 0.05, 1.0)
tail_sma = tail_df["sma"].to_numpy(dtype=float)

_, x0_slope = _weighted_line_fit(tail_sma, tail_df["x0_full"].to_numpy(dtype=float), tail_w)
_, y0_slope = _weighted_line_fit(tail_sma, tail_df["y0_full"].to_numpy(dtype=float), tail_w)
_, eps_slope = _weighted_line_fit(tail_sma, tail_df["eps"].to_numpy(dtype=float), tail_w)
_, pa_slope = _weighted_line_fit(tail_sma, tail_df["pa_unwrap"].to_numpy(dtype=float), tail_w)

x0_slope = _clip_slope(x0_slope, MODEL_CENTER_SLOPE_CLIP)
y0_slope = _clip_slope(y0_slope, MODEL_CENTER_SLOPE_CLIP)
eps_slope = _clip_slope(eps_slope, MODEL_EPS_SLOPE_CLIP)
pa_slope = _clip_slope(pa_slope, MODEL_PA_SLOPE_CLIP)

prof_tail_n = min(EXTRAP_PROFILE_FIT_N, len(fit_df))
prof_tail = fit_df.tail(prof_tail_n).copy()
prof_tail_w = np.clip(np.nan_to_num(prof_tail["real_data_frac"].to_numpy(dtype=float), nan=0.25), 0.05, 1.0)
outer_log_slope = 0.0
profile_extrap_note = "flat outer fallback"
if prof_tail_n >= EXTRAP_PROFILE_FIT_MIN_N and np.all(prof_tail["intens"].to_numpy(dtype=float) > 0.0):
    _, outer_log_slope = _weighted_line_fit(
        prof_tail["sma"].to_numpy(dtype=float),
        np.log(prof_tail["intens"].to_numpy(dtype=float)),
        prof_tail_w,
    )
    if np.isfinite(outer_log_slope) and outer_log_slope < 0.0:
        profile_extrap_note = f"log-linear outer tail, dlnI/dsma={outer_log_slope:.3e} pix^-1"
    else:
        outer_log_slope = 0.0

relax_scale_px = float(max(MODEL_SYNTH_RELAX_SCALE_PX, MODEL_SYNTH_SMA_STEP_PX))
max_delta_probe = max(lit_outer_r_px, relax_scale_px)
eps_outer_probe = float(np.clip(
    eps_edge + _saturating_extension(max_delta_probe, eps_slope, relax_scale_px),
    0.0,
    MODEL_FULL_EPS_CLIP_MAX,
))
q_outer_probe = max(MODEL_FULL_Q_MIN, 1.0 - eps_outer_probe)
sma_needed_for_outer_circle_px = float(lit_outer_r_px / q_outer_probe)
sma_model_full_max_px = float(max(fitted_sma_max_px, sma_needed_for_outer_circle_px) * SBF_MODEL_MARGIN)

sma_step_syn = float(max(MODEL_SYNTH_SMA_STEP_PX, 1.0))
outer_sma = np.arange(fitted_sma_max_px + sma_step_syn, sma_model_full_max_px + 0.5 * sma_step_syn, sma_step_syn)
if outer_sma.size and outer_sma[-1] < sma_model_full_max_px:
    outer_sma = np.append(outer_sma, sma_model_full_max_px)
elif not outer_sma.size and sma_model_full_max_px > fitted_sma_max_px + 0.5:
    outer_sma = np.array([sma_model_full_max_px], dtype=float)

delta_outer = np.maximum(outer_sma - fitted_sma_max_px, 0.0)
if outer_sma.size:
    x0_syn = x0_edge + _saturating_extension(delta_outer, x0_slope, relax_scale_px)
    y0_syn = y0_edge + _saturating_extension(delta_outer, y0_slope, relax_scale_px)
    eps_syn = np.clip(eps_edge + _saturating_extension(delta_outer, eps_slope, relax_scale_px), 0.0, MODEL_FULL_EPS_CLIP_MAX)
    pa_syn = pa_edge + _saturating_extension(delta_outer, pa_slope, relax_scale_px)
    int_syn = I_at_fit_edge * np.exp(outer_log_slope * delta_outer)
else:
    x0_syn = y0_syn = eps_syn = pa_syn = int_syn = np.array([], dtype=float)

fit_nodes = fit_df[["sma", "intens", "x0_full", "y0_full", "eps", "pa_unwrap"]].copy()
fit_nodes = fit_nodes.rename(columns={"pa_unwrap": "pa"})
fit_nodes["node_origin"] = "fitted"

if outer_sma.size:
    syn_nodes = pd.DataFrame({
        "sma": outer_sma,
        "intens": int_syn,
        "x0_full": x0_syn,
        "y0_full": y0_syn,
        "eps": eps_syn,
        "pa": pa_syn,
        "node_origin": "synthetic",
    })
    family_nodes = pd.concat([fit_nodes, syn_nodes], ignore_index=True)
else:
    family_nodes = fit_nodes.copy()

family_nodes = family_nodes.drop_duplicates(subset="sma", keep="last").sort_values("sma").reset_index(drop=True)
family_nodes = family_nodes[np.isfinite(family_nodes["intens"]) & (family_nodes["intens"] > 0.0)].reset_index(drop=True)
family_nodes["grad"] = _gradient_from_profile(
    family_nodes["sma"].to_numpy(dtype=float),
    family_nodes["intens"].to_numpy(dtype=float),
)
fit_nodes["grad"] = _gradient_from_profile(
    fit_nodes["sma"].to_numpy(dtype=float),
    fit_nodes["intens"].to_numpy(dtype=float),
)

fit_iso_full = IsophoteList([
    _SyntheticIso(row.sma, row.intens, row.eps, row.pa, row.x0_full, row.y0_full, row.grad)
    for row in fit_nodes.itertuples(index=False)
])
family_iso_full = IsophoteList([
    _SyntheticIso(row.sma, row.intens, row.eps, row.pa, row.x0_full, row.y0_full, row.grad)
    for row in family_nodes.itertuples(index=False)
])

model_fit_full = build_ellipse_model(img.shape, fit_iso_full, fill=np.nan)
model_fit_full = np.where(np.isfinite(model_fit_full) & (model_fit_full > 0.0), model_fit_full, np.nan)

model_full = build_ellipse_model(img.shape, family_iso_full, fill=np.nan)
model_full = np.where(np.isfinite(model_full) & (model_full > 0.0), model_full, np.nan)
model_full[(~signal_valid) | (~np.isfinite(img))] = np.nan
model_fit_full[(~signal_valid) | (~np.isfinite(img))] = np.nan

# Общие остатки до маскирования компактных источников.
# Оставляем NaN только вне валидной области гладкой модели.
остатки_общие = np.array(img - model_full, dtype=np.float32, copy=True)
остатки_общие[(~signal_valid) | (~np.isfinite(model_full)) | (~np.isfinite(img))] = np.nan

resid_full = np.array(остатки_общие, dtype=np.float32, copy=True)
resid_full[premask | (~np.isfinite(model_full)) | (~np.isfinite(img))] = np.nan

fitted_support_mask = np.isfinite(model_fit_full) & (model_fit_full > 0.0)
synthetic_support_mask = np.isfinite(model_full) & (model_full > 0.0) & (~fitted_support_mask)
n_fitted_support = int(fitted_support_mask.sum())
n_cont_only = int(synthetic_support_mask.sum())

x0_sbf_circ = x0_model_full
y0_sbf_circ = y0_model_full

def circular_annulus_model_coverage(r_in, r_out):
    yy_full, xx_full = np.indices(img.shape, dtype=float)
    rr = np.hypot(xx_full - x0_sbf_circ, yy_full - y0_sbf_circ)
    ann = (rr >= float(r_in)) & (rr <= float(r_out))
    valid_ann = ann & signal_valid & np.isfinite(img)
    model_ann = valid_ann & np.isfinite(model_full) & (model_full > 0.0)
    fitted_ann = model_ann & fitted_support_mask
    cont_ann = model_ann & synthetic_support_mask
    unmasked_model_ann = model_ann & (~premask)
    total_valid = int(valid_ann.sum())
    model_valid = int(model_ann.sum())
    fitted_valid = int(fitted_ann.sum())
    continuation_valid = int(cont_ann.sum())
    unmasked_model_valid = int(unmasked_model_ann.sum())
    frac_model = float(model_valid / total_valid) if total_valid > 0 else np.nan
    frac_unmasked_model = float(unmasked_model_valid / total_valid) if total_valid > 0 else np.nan
    frac_fitted = float(fitted_valid / total_valid) if total_valid > 0 else np.nan
    frac_cont = float(continuation_valid / total_valid) if total_valid > 0 else np.nan
    return total_valid, model_valid, unmasked_model_valid, frac_model, frac_unmasked_model, fitted_valid, continuation_valid, frac_fitted, frac_cont

inner_cov = circular_annulus_model_coverage(r1_in, r1_out)
outer_cov = circular_annulus_model_coverage(r2_in, r2_out)

print(
    f"[МОДЕЛЬ ПОЛНОГО КАДРА] fitted profile sma range = "
    f"{fitted_sma_min_px:.1f}..{fitted_sma_max_px:.1f} px "
    f"({fitted_sma_min_px * pix_scale:.2f}..{fitted_sma_max_px * pix_scale:.2f} arcsec)"
)
print(
    f"[МОДЕЛЬ ПОЛНОГО КАДРА] Jensen/TRGB-SBF circular annuli = "
    f"{SBF_LIT_INNER_ARCSEC[0]:.1f}-{SBF_LIT_INNER_ARCSEC[1]:.1f} arcsec and "
    f"{SBF_LIT_OUTER_ARCSEC[0]:.1f}-{SBF_LIT_OUTER_ARCSEC[1]:.1f} arcsec "
    f"= {r1_in:.1f}-{r1_out:.1f} px and {r2_in:.1f}-{r2_out:.1f} px"
)
print(
    f"[МОДЕЛЬ ПОЛНОГО КАДРА] trusted real-data support extends to sma≈{trusted_sma_max_px:.1f} px "
    f"({trusted_sma_max_px * pix_scale:.2f} arcsec) at real-data coverage >= {ISO_REAL_COVERAGE_MIN:.2f}"
)
print(
    f"[МОДЕЛЬ ПОЛНОГО КАДРА] fitted center median: x0={x0_model_full:.2f}, y0={y0_model_full:.2f}; "
    f"outer edge geometry: eps={eps_edge:.3f}, q={max(MODEL_FULL_Q_MIN, 1.0 - eps_edge):.3f}, pa={pa_edge:.3f} rad"
)
print(
    f"[МОДЕЛЬ ПОЛНОГО КАДРА] outer continuation slopes: dx0/dsma={x0_slope:.3e}, dy0/dsma={y0_slope:.3e}, "
    f"deps/dsma={eps_slope:.3e}, dpa/dsma={pa_slope:.3e}; relax_scale={relax_scale_px:.1f} px"
)
print(
    f"[МОДЕЛЬ ПОЛНОГО КАДРА] to cover circular R=32.8 arcsec, required elliptical sma≈"
    f"{sma_needed_for_outer_circle_px:.1f} px; built to {sma_model_full_max_px:.1f} px"
)
print(
    f"[МОДЕЛЬ ПОЛНОГО КАДРА] node counts: fitted={len(fit_nodes)}, synthetic={len(family_nodes) - len(fit_nodes)}, "
    f"family_total={len(family_nodes)}"
)
print(f"[МОДЕЛЬ ПОЛНОГО КАДРА] outer profile extrapolation: {profile_extrap_note}")
print(
    f"[МОДЕЛЬ ПОЛНОГО КАДРА] inner annulus model coverage: "
    f"valid={inner_cov[0]}, model={inner_cov[1]} ({100.0 * inner_cov[3]:.2f}%), "
    f"fitted={inner_cov[5]} ({100.0 * inner_cov[7]:.2f}%), "
    f"continuation={inner_cov[6]} ({100.0 * inner_cov[8]:.2f}%), "
    f"unmasked+model={inner_cov[2]} ({100.0 * inner_cov[4]:.2f}%)"
)
print(
    f"[МОДЕЛЬ ПОЛНОГО КАДРА] outer annulus model coverage: "
    f"valid={outer_cov[0]}, model={outer_cov[1]} ({100.0 * outer_cov[3]:.2f}%), "
    f"fitted={outer_cov[5]} ({100.0 * outer_cov[7]:.2f}%), "
    f"continuation={outer_cov[6]} ({100.0 * outer_cov[8]:.2f}%), "
    f"unmasked+model={outer_cov[2]} ({100.0 * outer_cov[4]:.2f}%)"
)
print(
    f"[МОДЕЛЬ ПОЛНОГО КАДРА] exact fitted-support pixels={n_fitted_support}, continuation-only pixels={n_cont_only}"
)
print(
    f"[МОДЕЛЬ ПОЛНОГО КАДРА] resid_full finite={int(np.isfinite(resid_full).sum())}, "
    f"model_full finite={int(np.isfinite(model_full).sum())}"
)
save_stage_table("07_isophotes", "all_fitted_points", iso_df)
save_stage_table("07_isophotes", "accepted_fitted_points", fit_df)
save_stage_table("07_isophotes", "full_family_nodes", family_nodes)


## Ограничение выбросов в остатках

Вычитаем гладкую модель и ограничиваем крайние значения на уровне заданного числа сигм. Это подавляет редкие сильные выбросы, не меняя выбранную область измерения.


In [ ]:
print("building sigma-capped full-frame science residual for SBF annuli...")

if "resid_full" not in globals() or "model_full" not in globals():
    raise RuntimeError("[ОГРАНИЧЕНИЕ ОСТАТКОВ] run the extrapolated full-frame model cell before sigma clipping")

science_clip_mask = (~premask) & np.isfinite(resid_full) & np.isfinite(model_full) & (model_full > 0.0)
n_science_clip = int(science_clip_mask.sum())
if n_science_clip < MIN_PIXELS_SBF:
    raise RuntimeError(f"[ОГРАНИЧЕНИЕ ОСТАТКОВ] too few valid science pixels: N={n_science_clip}")

vals_science_clip = resid_full[science_clip_mask]
mean_clip_full, med_clip_full, std_clip_full = sigma_clipped_stats(
    vals_science_clip,
    sigma=CLIP_SIGMA_QC,
    maxiters=CLIP_MAXIT_QC,
)

if (not np.isfinite(std_clip_full)) or std_clip_full <= 0.0:
    raise RuntimeError(f"[ОГРАНИЧЕНИЕ ОСТАТКОВ] bad clipped std={std_clip_full}")

clip_lo_full = float(med_clip_full - CLIP_SIGMA_QC * std_clip_full)
clip_hi_full = float(med_clip_full + CLIP_SIGMA_QC * std_clip_full)

resid_full_clip = np.array(resid_full, dtype=np.float32, copy=True)
resid_full_clip[science_clip_mask] = np.clip(
    resid_full_clip[science_clip_mask],
    clip_lo_full,
    clip_hi_full,
)
resid_full_clip[~science_clip_mask] = np.nan

resid_full_clip_delta = np.full_like(resid_full_clip, np.nan, dtype=np.float32)
resid_full_clip_delta[science_clip_mask] = (
    resid_full_clip[science_clip_mask] - resid_full[science_clip_mask]
)

n_clip_changed = int(np.count_nonzero(np.abs(resid_full_clip_delta[science_clip_mask]) > 0.0))
clip_tag = CLIP_TAG_QC

print(
    f"[ОГРАНИЧЕНИЕ ОСТАТКОВ] sigma={CLIP_SIGMA_QC}, valid={n_science_clip}, "
    f"med={med_clip_full:.3e}, std={std_clip_full:.3e}"
)
print(
    f"[ОГРАНИЧЕНИЕ ОСТАТКОВ] cap range = [{clip_lo_full:.3e}, {clip_hi_full:.3e}], "
    f"changed={n_clip_changed} px ({100.0 * n_clip_changed / n_science_clip:.2f}%)"
)


### Общие остатки до маскирования компактных источников

Это разность исходного рабочего кадра и гладкой модели до применения первичной и дополнительной масок. На ней шаровики ещё видны; массив сохраняется отдельно для визуальной проверки.


In [ ]:
if "остатки_общие" not in globals():
    raise RuntimeError("Общие остатки до маскирования ещё не построены")
print(f"[ОСТАТКИ] до маскирования компактных источников: {np.isfinite(остатки_общие).sum()} пикс.")


## Единая рабочая пара модели и остатков

Все дальнейшие измерения получают одну и ту же модель и один массив остатков. Это исключает случайное смешение разных промежуточных вариантов.


In [ ]:
print("selecting unified science residual/model path for all spectral SBF measurements...")

if "model_full" not in globals() or "resid_full_clip" not in globals():
    raise RuntimeError("[РАБОЧАЯ ВЕТВЬ] run the full-frame model and sigma-capped residual cells before spectral SBF")

science_model = model_full
science_resid = resid_full_clip
science_model_name = "model_full"
science_resid_name = f"resid_full_clip_{CLIP_TAG_QC}sigma"

if science_model.shape != science_resid.shape:
    raise RuntimeError(
        f"[РАБОЧАЯ ВЕТВЬ] shape mismatch: model={science_model.shape}, resid={science_resid.shape}"
    )

def _science_array_stats(arr):
    good = np.isfinite(arr)
    if not np.any(good):
        return {
            "shape": tuple(arr.shape),
            "n_finite": 0,
            "min": np.nan,
            "median": np.nan,
            "max": np.nan,
            "std": np.nan,
        }
    vals = arr[good]
    return {
        "shape": tuple(arr.shape),
        "n_finite": int(good.sum()),
        "min": float(np.nanmin(vals)),
        "median": float(np.nanmedian(vals)),
        "max": float(np.nanmax(vals)),
        "std": float(np.nanstd(vals)),
    }

science_model_stats = _science_array_stats(science_model)
science_resid_stats = _science_array_stats(science_resid)

print(f"[РАБОЧАЯ ВЕТВЬ] science_model = {science_model_name}")
print(f"[РАБОЧАЯ ВЕТВЬ] science_resid = {science_resid_name}")
print(
    f"[РАБОЧАЯ ВЕТВЬ][model] shape={science_model_stats['shape']}, finite={science_model_stats['n_finite']}, "
    f"min={science_model_stats['min']:.3e}, med={science_model_stats['median']:.3e}, "
    f"max={science_model_stats['max']:.3e}, std={science_model_stats['std']:.3e}"
)
print(
    f"[РАБОЧАЯ ВЕТВЬ][resid] shape={science_resid_stats['shape']}, finite={science_resid_stats['n_finite']}, "
    f"min={science_resid_stats['min']:.3e}, med={science_resid_stats['median']:.3e}, "
    f"max={science_resid_stats['max']:.3e}, std={science_resid_stats['std']:.3e}"
)


## Дополнительная маска по остаткам

После вычитания модели повторно ищем яркие компактные области, прежде всего пропущенные шаровые скопления. Маска расширяется на заданное число пикселей и объединяется с первичной.


In [ ]:
print("building supplemental compact residual mask from current science residual...")

if "science_resid" not in globals() or "science_model" not in globals():
    raise RuntimeError("[МАСКА ОСТАТКОВ] unified science residual/model are unavailable; rerun the science-path cell")
if "img" not in globals() or "model_full" not in globals():
    raise RuntimeError("[МАСКА ОСТАТКОВ] img/model_full are unavailable; rerun the model cell")
if "premask" not in globals():
    raise RuntimeError("[МАСКА ОСТАТКОВ] premask is unavailable; rerun the premask cell")

RESID_EXTRA_MASK_SIGMA = 3
RESID_EXTRA_MASK_NPIX = 9
RESID_EXTRA_MASK_KERNEL_SIGMA = 1.0
RESID_EXTRA_MASK_DILATE = 2
RESID_EXTRA_MASK_MAX_AREA = 4000
RESID_EXTRA_MASK_POSITIVE_ONLY = True
RESID_EXTRA_MASK_BRANCH = "compactresidmask"

if "premask_base" not in globals():
    premask_base = np.array(premask, dtype=bool, copy=True)
if "premask_src_base" not in globals() and "premask_src" in globals():
    premask_src_base = np.array(premask_src, dtype=bool, copy=True)

detect_support = (~premask_base) & np.isfinite(science_resid) & np.isfinite(science_model) & (science_model > 0.0)
n_detect_support = int(detect_support.sum())
if n_detect_support < MIN_PIXELS_SBF:
    raise RuntimeError(f"[МАСКА ОСТАТКОВ] too few science pixels for residual-mask detection: N={n_detect_support}")

vals_det = np.asarray(science_resid[detect_support], dtype=float)
_, med_det, std_det = sigma_clipped_stats(vals_det, sigma=CLIP_SIGMA_QC, maxiters=CLIP_MAXIT_QC)
if (not np.isfinite(std_det)) or std_det <= 0.0:
    raise RuntimeError(f"[МАСКА ОСТАТКОВ] bad residual std for compact-mask detection: std={std_det}")

resid_det = np.zeros(science_resid.shape, dtype=np.float32)
resid_centered = np.asarray(science_resid - med_det, dtype=np.float32)
if RESID_EXTRA_MASK_POSITIVE_ONLY:
    resid_det[detect_support] = np.maximum(resid_centered[detect_support], 0.0)
else:
    resid_det[detect_support] = np.abs(resid_centered[detect_support])

if RESID_EXTRA_MASK_KERNEL_SIGMA > 0.0:
    resid_det_smooth = gaussian_filter(resid_det, sigma=RESID_EXTRA_MASK_KERNEL_SIGMA, mode="nearest")
else:
    resid_det_smooth = resid_det

threshold_det = float(RESID_EXTRA_MASK_SIGMA * std_det)
segm_extra = detect_sources(resid_det_smooth, threshold_det, npixels=RESID_EXTRA_MASK_NPIX)

extra_resid_mask_raw = np.zeros_like(premask_base, dtype=bool)
extra_resid_mask = np.zeros_like(premask_base, dtype=bool)
df_resid_extra_sources = pd.DataFrame(columns=["label", "area_pix", "peak_resid", "peak_snr"])

if segm_extra is not None:
    seg_data = np.asarray(segm_extra.data, dtype=int)
    labels = np.unique(seg_data)
    labels = labels[labels > 0]
    if labels.size > 0:
        areas_all = np.bincount(seg_data.ravel())
        peak_vals = ndimage.maximum(resid_det, labels=seg_data, index=labels)
        peak_snr = peak_vals / std_det
        keep_rows = []
        keep_labels = []
        for lab, peak_val, peak_sig in zip(labels, peak_vals, peak_snr):
            area_pix = int(areas_all[int(lab)])
            if area_pix > RESID_EXTRA_MASK_MAX_AREA:
                continue
            if not np.isfinite(peak_val) or peak_val <= threshold_det:
                continue
            keep_labels.append(int(lab))
            keep_rows.append({
                "label": int(lab),
                "area_pix": area_pix,
                "peak_resid": float(peak_val),
                "peak_snr": float(peak_sig),
            })
        if keep_labels:
            extra_resid_mask_raw = np.isin(seg_data, np.asarray(keep_labels, dtype=int)) & detect_support
            if RESID_EXTRA_MASK_DILATE > 0:
                extra_resid_mask = ndimage.binary_dilation(extra_resid_mask_raw, iterations=RESID_EXTRA_MASK_DILATE)
            else:
                extra_resid_mask = extra_resid_mask_raw.copy()
            extra_resid_mask &= detect_support
            df_resid_extra_sources = pd.DataFrame(keep_rows).sort_values(["peak_snr", "area_pix"], ascending=[False, False]).reset_index(drop=True)

premask = np.array(premask_base | extra_resid_mask, dtype=bool)
premask_extra_resid = np.array(extra_resid_mask, dtype=bool, copy=True)

resid_full = np.array(img - model_full, dtype=np.float32, copy=True)
resid_full[premask | (~np.isfinite(model_full)) | (~np.isfinite(img))] = np.nan

science_clip_mask = (~premask) & np.isfinite(resid_full) & np.isfinite(model_full) & (model_full > 0.0)
n_science_clip = int(science_clip_mask.sum())
if n_science_clip < MIN_PIXELS_SBF:
    raise RuntimeError(f"[МАСКА ОСТАТКОВ] too few valid science pixels after extra mask: N={n_science_clip}")

vals_science_clip = np.asarray(resid_full[science_clip_mask], dtype=float)
mean_clip_full, med_clip_full, std_clip_full = sigma_clipped_stats(
    vals_science_clip,
    sigma=CLIP_SIGMA_QC,
    maxiters=CLIP_MAXIT_QC,
)
if (not np.isfinite(std_clip_full)) or std_clip_full <= 0.0:
    raise RuntimeError(f"[МАСКА ОСТАТКОВ] bad clipped std after extra mask: std={std_clip_full}")

clip_lo_full = float(med_clip_full - CLIP_SIGMA_QC * std_clip_full)
clip_hi_full = float(med_clip_full + CLIP_SIGMA_QC * std_clip_full)
resid_full_clip = np.array(resid_full, dtype=np.float32, copy=True)
resid_full_clip[science_clip_mask] = np.clip(resid_full_clip[science_clip_mask], clip_lo_full, clip_hi_full)
resid_full_clip[~science_clip_mask] = np.nan

resid_full_clip_delta = np.full_like(resid_full_clip, np.nan, dtype=np.float32)
resid_full_clip_delta[science_clip_mask] = resid_full_clip[science_clip_mask] - resid_full[science_clip_mask]

science_model = model_full
science_resid = resid_full_clip
science_model_name = "model_full"
science_resid_name = f"resid_full_clip_{CLIP_TAG_QC}sigma_plus_{RESID_EXTRA_MASK_BRANCH}"

n_raw_extra = int(extra_resid_mask_raw.sum())
n_final_extra = int(extra_resid_mask.sum())
n_clip_changed = int(np.count_nonzero(np.abs(resid_full_clip_delta[science_clip_mask]) > 0.0))



print(
    f"[МАСКА ОСТАТКОВ] residual sigma for compact leftover detection: med={med_det:.3e}, std={std_det:.3e}, "
    f"threshold={threshold_det:.3e} ({RESID_EXTRA_MASK_SIGMA:.1f} sigma)"
)
print(
    f"[МАСКА ОСТАТКОВ] kept compact residual labels={len(df_resid_extra_sources)}, raw_mask={n_raw_extra} px, "
    f"dilated_mask={n_final_extra} px, total premask coverage={100.0 * premask.sum() / premask.size:.2f}%"
)
print(
    f"[МАСКА ОСТАТКОВ] updated science clip: valid={n_science_clip}, med={med_clip_full:.3e}, std={std_clip_full:.3e}, "
    f"cap=[{clip_lo_full:.3e}, {clip_hi_full:.3e}], changed={n_clip_changed} px ({100.0 * n_clip_changed / n_science_clip:.2f}%)"
)
print("[МАСКА ОСТАТКОВ] рабочие остатки и общая маска обновлены.")
display(df_resid_extra_sources.head(20))


## Каталог компактных источников для $P_r$

Отдельно строим каталог источников, необходимых для оценки мощности неразрешённых примесей $P_r$. Этот каталог исправляет амплитуду SBF, но не участвует в построении гладкой модели.


In [ ]:
print("building compact-source catalog for unresolved-source variance correction...")

jy_per_pix = MJY_SR_TO_JY_PER_ARCSEC2 * pix_area
source_catalog_ref_mask = np.isfinite(model_full) & (model_full > 0.0)
source_catalog_columns = [
    "label", "xcentroid", "ycentroid", "area_pix",
    "flux_img_signed", "flux_img_positive", "flux_img_adopted",
    "flux_jy", "mag_ab",
]

if not np.any(source_catalog_ref_mask):
    raise RuntimeError("[КАТАЛОГ ИСТОЧНИКОВ] model_full has no positive pixels for reference mask")

source_catalog_seg_full = np.zeros(model_full.shape, dtype=np.int32)
source_catalog_build_info = {}
compact_source_catalog = None

yy_ref, xx_ref = np.where(source_catalog_ref_mask)
y0_src, y1_src = int(yy_ref.min()), int(yy_ref.max()) + 1
x0_src, x1_src = int(xx_ref.min()), int(xx_ref.max()) + 1
source_catalog_bbox = (y0_src, y1_src, x0_src, x1_src)

ref_mask_crop = np.asarray(source_catalog_ref_mask[y0_src:y1_src, x0_src:x1_src], dtype=bool)
catalog_image = np.array(
    img[y0_src:y1_src, x0_src:x1_src] - model_full[y0_src:y1_src, x0_src:x1_src],
    dtype=float,
    copy=True,
)
catalog_image[~np.isfinite(catalog_image)] = np.nan
catalog_image[~ref_mask_crop] = np.nan

valid_detect = ref_mask_crop & np.isfinite(catalog_image)
detect_values = catalog_image[valid_detect]
if detect_values.size < MIN_PIXELS_SIGMA_CLIP:
    raise RuntimeError(f"[КАТАЛОГ ИСТОЧНИКОВ] too few valid residual pixels in model-support crop: N={detect_values.size}")

mean_det, med_det, std_det = sigma_clipped_stats(
    detect_values,
    sigma=SIGMA_STAT,
    maxiters=SIGMA_MAXIT,
)
threshold_det = float(med_det + SBF_PR_DET_SIGMA * max(std_det, ROBUST_SCALE_FLOOR))

detect_image = np.zeros_like(catalog_image, dtype=float)
detect_image[valid_detect] = catalog_image[valid_detect]

segm_detect = detect_sources(
    detect_image,
    threshold=threshold_det,
    npixels=SBF_PR_DET_NPIXELS,
    connectivity=SOURCE_DETECT_CONNECTIVITY,
)
n_detect_raw = int(segm_detect.nlabels) if segm_detect is not None else 0

if segm_detect is not None and SBF_PR_DO_DEBLEND:
    try:
        segm_detect = deblend_sources(
            detect_image,
            segm_detect,
            npixels=SBF_PR_DET_NPIXELS,
            nlevels=DEBLEND_NLEVELS,
            contrast=DEBLEND_CONTRAST,
            nproc=DEBLEND_NPROC,
            progress_bar=False,
        )
    except Exception as err:
        print(f"[КАТАЛОГ ИСТОЧНИКОВ] deblend skipped: {err}")

n_detect_deblend = int(segm_detect.nlabels) if segm_detect is not None else 0

premask_overlap_count = np.nan
if "premask_segm" in globals() and premask_segm is not None and "premask_compact_labels" in globals():
    premask_crop = np.asarray(premask_segm.data[y0_src:y1_src, x0_src:x1_src], dtype=int)
    premask_overlap_labels = np.intersect1d(
        np.unique(premask_crop[ref_mask_crop & (premask_crop > 0)]),
        np.asarray(premask_compact_labels, dtype=int),
        assume_unique=False,
    )
    premask_overlap_count = int(premask_overlap_labels.size)

source_catalog_df = pd.DataFrame(columns=source_catalog_columns)
source_catalog_seg_crop = np.zeros_like(detect_image, dtype=np.int32)
dropped_large_area = 0
dropped_nonpositive_signed_flux = 0
dropped_nonfinite_mag = 0
n_compact_area = 0
n_final = 0
source_pixels_final = 0

if segm_detect is not None and segm_detect.nlabels > 0:
    seg_data = np.asarray(segm_detect.data, dtype=np.int32)
    seg_data[~ref_mask_crop] = 0

    labels_all = np.asarray(segm_detect.labels, dtype=int)
    max_label = int(seg_data.max()) if seg_data.size else 0
    counts = np.bincount(seg_data.ravel(), minlength=max_label + 1) if max_label > 0 else np.zeros(1, dtype=int)
    counts[0] = 0

    compact_labels = labels_all[(counts[labels_all] > 0) & (counts[labels_all] <= SBF_PR_MAX_COMPACT_AREA)]
    dropped_large_area = int(labels_all.size - compact_labels.size)
    n_compact_area = int(compact_labels.size)

    label_lut = np.zeros(max_label + 1, dtype=np.int32)
    if compact_labels.size > 0:
        label_lut[compact_labels] = compact_labels.astype(np.int32)
    source_catalog_seg_crop = label_lut[seg_data]

    ys_src, xs_src = np.nonzero(source_catalog_seg_crop > 0)
    if ys_src.size > 0:
        labels_src = source_catalog_seg_crop[ys_src, xs_src].astype(np.int64, copy=False)
        labels_present = np.unique(labels_src)
        values_src = np.nan_to_num(catalog_image[ys_src, xs_src], nan=0.0)
        positive_values_src = np.clip(values_src, 0.0, None)

        signed_sum = np.bincount(labels_src, weights=values_src, minlength=max_label + 1)
        positive_sum = np.bincount(labels_src, weights=positive_values_src, minlength=max_label + 1)
        area_sum = np.bincount(labels_src, minlength=max_label + 1)
        x_sum = np.bincount(labels_src, weights=xs_src.astype(float), minlength=max_label + 1)
        y_sum = np.bincount(labels_src, weights=ys_src.astype(float), minlength=max_label + 1)

        rows = []
        kept_labels = []
        for label in labels_present.astype(int):
            area_pix = int(area_sum[label])
            if area_pix <= 0:
                continue

            flux_img_signed = float(signed_sum[label])
            flux_img_positive = float(positive_sum[label])
            flux_img_adopted = flux_img_signed

            if not (np.isfinite(flux_img_adopted) and flux_img_adopted > 0.0):
                dropped_nonpositive_signed_flux += 1
                continue

            flux_jy = float(flux_img_adopted * jy_per_pix)
            mag_ab = float(-2.5 * np.log10(flux_jy / AB_ZEROPOINT_JY)) if flux_jy > 0.0 else np.nan
            if not np.isfinite(mag_ab):
                dropped_nonfinite_mag += 1
                continue

            kept_labels.append(label)
            rows.append({
                "label": label,
                "xcentroid": float(x0_src + x_sum[label] / area_pix),
                "ycentroid": float(y0_src + y_sum[label] / area_pix),
                "area_pix": area_pix,
                "flux_img_signed": flux_img_signed,
                "flux_img_positive": flux_img_positive,
                "flux_img_adopted": flux_img_adopted,
                "flux_jy": flux_jy,
                "mag_ab": mag_ab,
            })

        source_catalog_df = pd.DataFrame(rows, columns=source_catalog_columns)
        if not source_catalog_df.empty:
            source_catalog_df = source_catalog_df.sort_values(["mag_ab", "label"], na_position="last").reset_index(drop=True)

        final_lut = np.zeros(max_label + 1, dtype=np.int32)
        if kept_labels:
            final_lut[np.asarray(kept_labels, dtype=int)] = np.asarray(kept_labels, dtype=np.int32)
        source_catalog_seg_crop = final_lut[source_catalog_seg_crop]
        source_catalog_seg_full[y0_src:y1_src, x0_src:x1_src] = source_catalog_seg_crop
        n_final = int(len(kept_labels))
        source_pixels_final = int(np.count_nonzero(source_catalog_seg_crop))

source_catalog_build_info.update({
    "bbox": source_catalog_bbox,
    "support_pixels": int(source_catalog_ref_mask.sum()),
    "n_detect_raw": int(n_detect_raw),
    "n_detect_deblend": int(n_detect_deblend),
    "n_compact_area": int(n_compact_area),
    "dropped_large_area": int(dropped_large_area),
    "dropped_nonpositive_signed_flux": int(dropped_nonpositive_signed_flux),
    "dropped_nonfinite_mag": int(dropped_nonfinite_mag),
    "n_final": int(n_final),
    "source_pixels_final": int(source_pixels_final),
    "premask_overlap_count": premask_overlap_count,
    "detect_threshold": float(threshold_det),
    "detect_med": float(med_det),
    "detect_std": float(std_det),
    "detect_image": SBF_PR_SOURCE_IMAGE,
})

source_catalog_csv_path = save_stage_table("09_sources", "compact_source_catalog", source_catalog_df)

finite_mag = np.isfinite(source_catalog_df.get("mag_ab", pd.Series(dtype=float))).sum() if not source_catalog_df.empty else 0
print(
    f"[КАТАЛОГ ИСТОЧНИКОВ] detect image={SBF_PR_SOURCE_IMAGE}, crop={y0_src}:{y1_src}, {x0_src}:{x1_src}, "
    f"support pixels={int(source_catalog_ref_mask.sum())}"
)
print(
    f"[КАТАЛОГ ИСТОЧНИКОВ] residual stats in support: med={med_det:.3e}, std={std_det:.3e}, "
    f"threshold={threshold_det:.3e}"
)
if np.isfinite(premask_overlap_count):
    print(f"[КАТАЛОГ ИСТОЧНИКОВ] old premask-overlap inside model support = {int(premask_overlap_count)} compact labels")
print(
    f"[КАТАЛОГ ИСТОЧНИКОВ] segmentation: raw={n_detect_raw}, deblended={n_detect_deblend}, "
    f"compact-area pass={n_compact_area}, dropped_large={dropped_large_area}"
)
print(
    f"[КАТАЛОГ ИСТОЧНИКОВ] photometry filter: kept={n_final}, dropped_nonpositive_signed_flux={dropped_nonpositive_signed_flux}, "
    f"dropped_nonfinite_mag={dropped_nonfinite_mag}, source pixels kept={source_pixels_final}"
)
print(
    f"[КАТАЛОГ ИСТОЧНИКОВ] compact catalog size={len(source_catalog_df)}, finite mags={finite_mag}, "
    f"reference pixels={int(source_catalog_ref_mask.sum())}"
)
print(f"[OUT] compact source catalog -> {source_catalog_csv_path}")


## Оценка мощности неразрешённых источников

Аппроксимируем функцию светимости примесей и вычисляем $P_r$. Исправленная мощность флуктуаций равна $P_{fluc}=P_0-P_r$.


In [ ]:
print("preparing unresolved-source variance helpers...")

def sbf_ab_zeropoint_from_pix_area(pix_area):
    jy_per_pix_local = MJY_SR_TO_JY_PER_ARCSEC2 * pix_area
    return float(-2.5 * np.log10(jy_per_pix_local / AB_ZEROPOINT_JY))

def sbf_flux_image_units_from_mag(mag_ab, pix_area):
    zp_ab_local = sbf_ab_zeropoint_from_pix_area(pix_area)
    return float(10.0 ** (-0.4 * (float(mag_ab) - zp_ab_local)))

def optional_finite_float(value):
    try:
        value_f = float(value)
    except (TypeError, ValueError):
        return np.nan
    return value_f if np.isfinite(value_f) else np.nan

def select_sources_in_region_mask(region_mask, region_origin_x=0, region_origin_y=0):
    if source_catalog_df.empty:
        return source_catalog_df.copy()

    region_mask = np.asarray(region_mask, dtype=bool)
    ny_mask, nx_mask = region_mask.shape

    seg_full = globals().get("source_catalog_seg_full", None)
    if isinstance(seg_full, np.ndarray) and seg_full.ndim == 2:
        y0 = int(region_origin_y)
        x0 = int(region_origin_x)
        y1 = y0 + ny_mask
        x1 = x0 + nx_mask
        if 0 <= y0 < y1 <= seg_full.shape[0] and 0 <= x0 < x1 <= seg_full.shape[1]:
            seg_view = np.asarray(seg_full[y0:y1, x0:x1], dtype=int)
            labels_in_region = np.unique(seg_view[region_mask & (seg_view > 0)])
            if labels_in_region.size == 0:
                return source_catalog_df.iloc[0:0].copy()
            keep = source_catalog_df["label"].isin(labels_in_region.astype(int))
            return source_catalog_df.loc[keep].copy().reset_index(drop=True)

    keep = []
    for idx, row in source_catalog_df.iterrows():
        xcen = row.get("xcentroid", np.nan)
        ycen = row.get("ycentroid", np.nan)
        if not (np.isfinite(xcen) and np.isfinite(ycen)):
            continue

        ix = int(np.rint(float(xcen))) - int(region_origin_x)
        iy = int(np.rint(float(ycen))) - int(region_origin_y)
        if 0 <= ix < nx_mask and 0 <= iy < ny_mask and bool(region_mask[iy, ix]):
            keep.append(idx)

    if not keep:
        return source_catalog_df.iloc[0:0].copy()
    return source_catalog_df.loc[keep].copy().reset_index(drop=True)

def summarize_sources_in_region_mask(region_mask, region_origin_x=0, region_origin_y=0):
    reg_sources = select_sources_in_region_mask(
        region_mask,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )
    if reg_sources.empty:
        return {
            "n_overlap": 0,
            "n_valid": 0,
            "n_nonpositive_flux": 0,
            "mag_min": np.nan,
            "mag_med": np.nan,
            "mag_max": np.nan,
        }

    flux = reg_sources["flux_jy"].to_numpy(dtype=float) if "flux_jy" in reg_sources else np.full(len(reg_sources), np.nan)
    mags = reg_sources["mag_ab"].to_numpy(dtype=float) if "mag_ab" in reg_sources else np.full(len(reg_sources), np.nan)
    valid = np.isfinite(flux) & (flux > SBF_PR_CATALOG_MIN_FLUX_JY) & np.isfinite(mags)
    mags_valid = mags[valid]

    return {
        "n_overlap": int(len(reg_sources)),
        "n_valid": int(np.count_nonzero(valid)),
        "n_nonpositive_flux": int(np.count_nonzero(~(np.isfinite(flux) & (flux > SBF_PR_CATALOG_MIN_FLUX_JY)))),
        "mag_min": float(np.nanmin(mags_valid)) if mags_valid.size > 0 else np.nan,
        "mag_med": float(np.nanmedian(mags_valid)) if mags_valid.size > 0 else np.nan,
        "mag_max": float(np.nanmax(mags_valid)) if mags_valid.size > 0 else np.nan,
    }

def estimate_turnover_magnitude(mags, mag_bin=None):
    if mag_bin is None:
        mag_bin = SBF_PR_MAG_BIN
    arr = np.asarray(mags, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size < 3:
        return np.nan, {"method": "too_few_sources", "n_sources": int(arr.size)}

    lo = float(np.floor(np.nanmin(arr) / mag_bin) * mag_bin)
    hi = float(np.ceil(np.nanmax(arr) / mag_bin) * mag_bin + mag_bin)
    edges = np.arange(lo, hi + 0.5 * mag_bin, mag_bin)
    if edges.size < 3:
        return np.nan, {"method": "bad_edges", "n_sources": int(arr.size)}

    counts, edges = np.histogram(arr, bins=edges)
    if not np.any(counts > 0):
        return np.nan, {"method": "empty_hist", "n_sources": int(arr.size)}

    idx_peak = int(np.argmax(counts))
    m_turn = float(0.5 * (edges[idx_peak] + edges[idx_peak + 1]))
    return m_turn, {
        "method": "turnover_hist",
        "n_sources": int(arr.size),
        "peak_count": int(counts[idx_peak]),
        "n_bins": int(counts.size),
    }

def fit_combined_contaminant_lf(mags, area_pix, m_lim, fit_width=None, mag_bin=None, gamma_fallback=None):
    if fit_width is None:
        fit_width = SBF_PR_FIT_WIDTH
    if mag_bin is None:
        mag_bin = SBF_PR_MAG_BIN
    if gamma_fallback is None or not np.isfinite(gamma_fallback):
        gamma_fallback = SBF_PR_DEFAULT_GAMMA

    arr = np.asarray(mags, dtype=float)
    arr = arr[np.isfinite(arr)]
    area_pix = float(area_pix)
    out = {
        "lf_model": "combined_power_law",
        "lf_gamma": np.nan,
        "lf_phi_mlim": np.nan,
        "lf_fit_method": "unavailable",
        "lf_fit_rmse": np.nan,
        "lf_n_fit_bins": 0,
        "lf_n_fit_sources": 0,
        "lf_fit_mbright": float(m_lim - fit_width),
        "lf_fit_mfaint": float(m_lim),
    }

    if arr.size == 0 or (not np.isfinite(m_lim)) or area_pix <= 0.0:
        out["lf_fit_method"] = "no_sources_or_bad_limit"
        return out

    m_lo = float(m_lim - fit_width)
    fit_sel = (arr >= m_lo) & (arr <= m_lim)
    fit_mags = arr[fit_sel]
    out["lf_n_fit_sources"] = int(fit_mags.size)

    edges = np.arange(m_lo, m_lim + mag_bin * 1.01, mag_bin)
    if edges.size < 3:
        out["lf_fit_method"] = "bad_edges"
        return out

    counts, edges = np.histogram(fit_mags, bins=edges)
    centers = 0.5 * (edges[:-1] + edges[1:])
    positive = counts > 0
    out["lf_n_fit_bins"] = int(np.count_nonzero(positive))

    gamma_min, gamma_max = map(float, SBF_PR_GAMMA_BOUNDS)
    local_fit_ok = out["lf_n_fit_bins"] >= int(SBF_PR_MIN_FIT_BINS)

    if local_fit_ok:
        x = centers[positive]
        y = np.log10(counts[positive] / (area_pix * mag_bin))
        w = np.sqrt(counts[positive].astype(float))
        try:
            coeff = np.polyfit(x, y, 1, w=w)
            gamma = float(coeff[0])
            intercept = float(coeff[1])
            if np.isfinite(gamma) and gamma_min <= gamma <= gamma_max:
                y_fit = intercept + gamma * x
                rmse = float(np.sqrt(np.nanmean((y - y_fit) ** 2))) if x.size > 0 else np.nan
                phi_mlim = float(10.0 ** (intercept + gamma * m_lim))
                if np.isfinite(phi_mlim) and phi_mlim > 0.0:
                    out.update({
                        "lf_gamma": gamma,
                        "lf_phi_mlim": phi_mlim,
                        "lf_fit_method": "local_histogram",
                        "lf_fit_rmse": rmse,
                    })
                    return out
        except Exception as err:
            out["lf_fit_method"] = f"local_histogram_failed: {err}"

    gamma = float(np.clip(gamma_fallback, gamma_min, gamma_max))
    span_integral = (1.0 - 10.0 ** (-gamma * fit_width)) / (gamma * np.log(10.0)) if gamma > 0.0 else float(fit_width)
    n_window = int(np.count_nonzero(fit_sel))
    if area_pix > 0.0 and span_integral > 0.0 and n_window > 0:
        phi_mlim = float((n_window / area_pix) / span_integral)
        out.update({
            "lf_gamma": gamma,
            "lf_phi_mlim": phi_mlim,
            "lf_fit_method": "local_window_norm_fixed_gamma",
        })
    else:
        out.update({
            "lf_gamma": gamma,
            "lf_fit_method": "fixed_gamma_no_local_norm",
        })
    return out

def integrate_pr_from_powerlaw_lf(phi_mlim, gamma, m_lim, pix_area):
    if not (np.isfinite(phi_mlim) and phi_mlim > 0.0 and np.isfinite(gamma) and np.isfinite(m_lim)):
        return np.nan
    if gamma >= 0.8:
        return np.nan

    f_lim_img = sbf_flux_image_units_from_mag(m_lim, pix_area)
    return float(phi_mlim * (f_lim_img ** 2) / ((0.8 - gamma) * np.log(10.0)))

def build_global_pr_reference():
    if source_catalog_df.empty:
        return {
            "region": "global_science_reference",
            "n_detected_sources": 0,
            "area_pix": int(source_catalog_ref_mask.sum()),
            "m_lim": np.nan,
            "m_lim_method": "empty_catalog",
            "lf_gamma": np.nan,
            "lf_phi_mlim": np.nan,
            "lf_fit_method": "empty_catalog",
            "Pr": np.nan,
        }

    ref_sources = select_sources_in_region_mask(source_catalog_ref_mask, region_origin_x=0, region_origin_y=0)
    ref_sources = ref_sources[
        np.isfinite(ref_sources.get("mag_ab", np.nan))
        & np.isfinite(ref_sources.get("flux_jy", np.nan))
        & (ref_sources.get("flux_jy", np.nan) > SBF_PR_CATALOG_MIN_FLUX_JY)
    ].copy()

    ref_mags = ref_sources["mag_ab"].to_numpy(dtype=float) if not ref_sources.empty else np.array([], dtype=float)
    m_lim_override = optional_finite_float(SBF_PR_MLIM_OVERRIDE)
    if np.isfinite(m_lim_override):
        m_lim = float(m_lim_override)
        m_lim_method = "manual_override"
    else:
        m_turn, turn_info = estimate_turnover_magnitude(ref_mags)
        if np.isfinite(m_turn) and ref_mags.size >= SBF_PR_MIN_SOURCES_GLOBAL:
            m_lim = float(m_turn + SBF_PR_MLIM_OFFSET)
            m_lim_method = f"global_{turn_info.get('method', 'turnover')}"
        elif ref_mags.size > 0:
            m_lim = float(np.nanquantile(ref_mags, SBF_PR_FALLBACK_QUANTILE) + SBF_PR_MLIM_OFFSET)
            m_lim_method = "global_quantile_fallback"
        else:
            m_lim = np.nan
            m_lim_method = "no_global_sources"

    fit = fit_combined_contaminant_lf(
        ref_mags,
        area_pix=float(source_catalog_ref_mask.sum()),
        m_lim=m_lim,
        gamma_fallback=SBF_PR_DEFAULT_GAMMA,
    )
    pr_value = integrate_pr_from_powerlaw_lf(fit.get("lf_phi_mlim", np.nan), fit.get("lf_gamma", np.nan), m_lim, pix_area)

    out = {
        "region": "global_science_reference",
        "n_detected_sources": int(ref_mags.size),
        "area_pix": int(source_catalog_ref_mask.sum()),
        "m_lim": m_lim,
        "m_lim_method": m_lim_method,
        "Pr": pr_value,
        **fit,
    }
    return out

def estimate_pr_for_region(region_mask, model, region_name="region", region_origin_x=0, region_origin_y=0):
    if not SBF_PR_ENABLE:
        return {
            "Pr": 0.0,
            "Pr_over_P0": np.nan,
            "n_detected_sources": 0,
            "n_detected_sources_total": 0,
            "m_lim": np.nan,
            "m_lim_method": "disabled",
            "lf_model": "disabled",
            "lf_gamma": np.nan,
            "lf_phi_mlim": np.nan,
            "lf_fit_method": "disabled",
            "lf_fit_rmse": np.nan,
            "lf_n_fit_bins": 0,
            "lf_n_fit_sources": 0,
            "lf_fit_mbright": np.nan,
            "lf_fit_mfaint": np.nan,
            "pr_note": "Pr correction disabled",
        }

    analysis_mask = np.asarray(region_mask, dtype=bool) & np.isfinite(model) & (model > 0.0)
    area_pix = int(np.count_nonzero(analysis_mask))
    reg_sources = select_sources_in_region_mask(analysis_mask, region_origin_x=region_origin_x, region_origin_y=region_origin_y)
    reg_sources_valid = reg_sources[
        np.isfinite(reg_sources.get("mag_ab", np.nan))
        & np.isfinite(reg_sources.get("flux_jy", np.nan))
        & (reg_sources.get("flux_jy", np.nan) > SBF_PR_CATALOG_MIN_FLUX_JY)
    ].copy()
    reg_mags = reg_sources_valid["mag_ab"].to_numpy(dtype=float) if not reg_sources_valid.empty else np.array([], dtype=float)

    global_ref = globals().get("sbf_pr_global_reference", None)
    global_gamma = global_ref.get("lf_gamma", np.nan) if isinstance(global_ref, dict) else np.nan
    global_phi = global_ref.get("lf_phi_mlim", np.nan) if isinstance(global_ref, dict) else np.nan
    global_mlim = global_ref.get("m_lim", np.nan) if isinstance(global_ref, dict) else np.nan

    m_lim_override = optional_finite_float(SBF_PR_MLIM_OVERRIDE)
    if np.isfinite(m_lim_override):
        m_lim = float(m_lim_override)
        m_lim_method = "manual_override"
    else:
        m_turn_local, local_turn_info = estimate_turnover_magnitude(reg_mags)
        if np.isfinite(m_turn_local) and reg_mags.size >= SBF_PR_MIN_SOURCES_REGION:
            m_lim = float(m_turn_local + SBF_PR_MLIM_OFFSET)
            m_lim_method = f"region_{local_turn_info.get('method', 'turnover')}"
        elif np.isfinite(global_mlim):
            m_lim = float(global_mlim)
            m_lim_method = "global_reference_turnover"
        elif reg_mags.size > 0:
            m_lim = float(np.nanquantile(reg_mags, SBF_PR_FALLBACK_QUANTILE) + SBF_PR_MLIM_OFFSET)
            m_lim_method = "region_quantile_fallback"
        else:
            m_lim = np.nan
            m_lim_method = "no_region_sources"

    fit = fit_combined_contaminant_lf(
        reg_mags,
        area_pix=float(area_pix),
        m_lim=m_lim,
        gamma_fallback=global_gamma,
    )

    pr_note = []
    if (not np.isfinite(fit.get("lf_phi_mlim", np.nan)) or fit.get("lf_phi_mlim", np.nan) <= 0.0) and np.isfinite(global_phi) and global_phi > 0.0:
        fit["lf_phi_mlim"] = float(global_phi)
        fit["lf_fit_method"] = str(fit.get("lf_fit_method", "unavailable")) + "+global_phi"
        pr_note.append("used global LF normalization")

    pr_value = integrate_pr_from_powerlaw_lf(fit.get("lf_phi_mlim", np.nan), fit.get("lf_gamma", np.nan), m_lim, pix_area)
    if np.isfinite(pr_value) and np.isfinite(global_ref.get("Pr", np.nan) if isinstance(global_ref, dict) else np.nan):
        pass
    elif np.isfinite(global_ref.get("Pr", np.nan) if isinstance(global_ref, dict) else np.nan):
        pr_value = float(global_ref["Pr"])
        pr_note.append("used global Pr fallback")

    if np.isfinite(pr_value) and np.isfinite(area_pix) and area_pix > 0 and reg_mags.size == 0:
        pr_note.append("no local detected sources; correction inherited from global reference")

    if np.isfinite(pr_value) and pr_value < 0.0:
        pr_value = np.nan
        pr_note.append("negative Pr clipped to NaN")

    if not pr_note:
        pr_note.append("local combined contaminant LF")

    return {
        "Pr": pr_value,
        "Pr_over_P0": np.nan,
        "n_detected_sources": int(reg_mags.size),
        "n_detected_sources_total": int(len(reg_sources)),
        "m_lim": m_lim,
        "m_lim_method": m_lim_method,
        "lf_model": fit.get("lf_model", "combined_power_law"),
        "lf_gamma": fit.get("lf_gamma", np.nan),
        "lf_phi_mlim": fit.get("lf_phi_mlim", np.nan),
        "lf_fit_method": fit.get("lf_fit_method", "unavailable"),
        "lf_fit_rmse": fit.get("lf_fit_rmse", np.nan),
        "lf_n_fit_bins": fit.get("lf_n_fit_bins", 0),
        "lf_n_fit_sources": fit.get("lf_n_fit_sources", 0),
        "lf_fit_mbright": fit.get("lf_fit_mbright", np.nan),
        "lf_fit_mfaint": fit.get("lf_fit_mfaint", np.nan),
        "pr_note": "; ".join(pr_note),
    }

def solve_sbf_power_budget(P0, Imean, Pr, pix_area, P0_fit_sigma=np.nan):
    out = {
        "measurement_ok": False,
        "failure_reason": "",
        "P0": float(P0) if np.isfinite(P0) else np.nan,
        "Pr": float(Pr) if np.isfinite(Pr) else np.nan,
        "P_fluc": np.nan,
        "Pr_over_P0": np.nan,
        "Imean": float(Imean) if np.isfinite(Imean) else np.nan,
        "Pf_spec_raw": np.nan,
        "Pf_spec": np.nan,
        "mbar_spec_raw": np.nan,
        "mbar_spec": np.nan,
        "Pf_spec_sigma_formal": np.nan,
        "Pf_spec_sigma_formal_raw": np.nan,
        "mbar_fit_sigma": np.nan,
        "mbar_fit_sigma_raw": np.nan,
    }

    if not (np.isfinite(P0) and P0 > 0.0):
        out["failure_reason"] = f"bad raw P0={P0}"
        return out
    if not (np.isfinite(Imean) and Imean > 0.0):
        out["failure_reason"] = f"bad Imean={Imean}"
        return out

    zp_ab_local = sbf_ab_zeropoint_from_pix_area(pix_area)
    Pf_spec_raw = float(P0 / Imean)
    out["Pf_spec_raw"] = Pf_spec_raw
    if np.isfinite(Pf_spec_raw) and Pf_spec_raw > 0.0:
        out["mbar_spec_raw"] = float(-2.5 * np.log10(Pf_spec_raw) + zp_ab_local)
        if np.isfinite(P0_fit_sigma) and P0_fit_sigma > 0.0:
            out["Pf_spec_sigma_formal_raw"] = float(Pf_spec_raw * P0_fit_sigma / P0)
            out["mbar_fit_sigma_raw"] = float((2.5 / np.log(10.0)) * P0_fit_sigma / P0)

    if not np.isfinite(Pr):
        out["failure_reason"] = "Pr is not finite"
        return out

    P_fluc = float(P0 - Pr)
    out["P_fluc"] = P_fluc
    out["Pr_over_P0"] = float(Pr / P0) if np.isfinite(Pr) else np.nan
    if not (np.isfinite(P_fluc) and P_fluc > 0.0):
        out["failure_reason"] = f"non-positive corrected power: P0={P0:.3e}, Pr={Pr:.3e}, P_fluc={P_fluc:.3e}"
        return out

    Pf_spec = float(P_fluc / Imean)
    out["Pf_spec"] = Pf_spec
    if not (np.isfinite(Pf_spec) and Pf_spec > 0.0):
        out["failure_reason"] = f"bad corrected Pf_spec={Pf_spec}"
        return out

    out["mbar_spec"] = float(-2.5 * np.log10(Pf_spec) + zp_ab_local)
    if np.isfinite(P0_fit_sigma) and P0_fit_sigma > 0.0:
        out["Pf_spec_sigma_formal"] = float(Pf_spec * P0_fit_sigma / P_fluc)
        out["mbar_fit_sigma"] = float((2.5 / np.log(10.0)) * P0_fit_sigma / P_fluc)

    out["measurement_ok"] = True
    return out

sbf_pr_global_reference = build_global_pr_reference()
print(
    f"[Pr-GLOBAL] Nsrc={sbf_pr_global_reference.get('n_detected_sources', 0)}, "
    f"m_lim={sbf_pr_global_reference.get('m_lim', np.nan):.3f}, "
    f"gamma={sbf_pr_global_reference.get('lf_gamma', np.nan):.3f}, "
    f"Pr={sbf_pr_global_reference.get('Pr', np.nan):.3e}, "
    f"fit={sbf_pr_global_reference.get('lf_fit_method', 'n/a')}"
)


## Пять проверочных FITS

Сохраняем гладкую модель, модель только по найденным изофотам, общие остатки до всех масок компактных источников, рабочие остатки после масок, а ниже — рабочие остатки в объединении двух колец.


In [ ]:
print("Сохраняю четыре полнокадровых проверочных FITS...")

def сохранить_скалярный_fits(путь, данные, заголовок, описание):
    кадр = np.squeeze(np.asarray(данные, dtype=np.float32))
    if кадр.ndim != 2:
        raise ValueError(f"Проверочный FITS должен быть двумерным скалярным кадром, получено {кадр.shape}")
    hdr = заголовок.copy()
    hdr.remove("EXTNAME", ignore_missing=True, remove_all=True)
    hdr["SBFPLANE"] = ("SCALAR", "two-dimensional scalar image")
    hdr["SBFVIEW"] = ("GRAYSCALE", "recommended display LUT")
    hdr.add_history(описание)
    fits.PrimaryHDU(data=кадр, header=hdr).writeto(путь, overwrite=True)
    with fits.open(путь, memmap=True) as проверка:
        if len(проверка) != 1 or проверка[0].data is None or проверка[0].data.ndim != 2:
            raise RuntimeError(f"Нарушен контракт двумерного FITS после записи: {путь}")

проверочные_кадры = {
    "01_модель_чистая": model_full,
    "02_изофоты_чистые": model_fit_full,
    "03_остатки_общие": остатки_общие,
    "04_остатки_общие_рабочие": science_resid,
}
описания_fits = {
    "01_модель_чистая": "Clean smooth galaxy model",
    "02_изофоты_чистые": "Clean fitted-isophote model",
    "03_остатки_общие": "Full residual before compact-source masks",
    "04_остатки_общие_рабочие": "Working full residual after compact-source masks",
}

пути_проверочных_кадров = {}
for название, данные in проверочные_кадры.items():
    путь = out_dir / f"{stem}_{название}.fits"
    сохранить_скалярный_fits(путь, данные, output_header, описания_fits[название])
    пути_проверочных_кадров[название] = путь
    print(f"[FITS] {название} -> {путь}")


## Спектральное измерение SBF

Во Фурье-пространстве аппроксимируем $P(k)=P_0E(k)+P_1$, где $E(k)$ строится по PSF, гладкой модели и маске. Затем вычитаем вклад $P_r$ и переводим мощность в видимую звёздную величину SBF.


## Функция рассеяния точки

Строим PSF через локальные данные `stpsf` и WSS OPD, приводим её к пиксельной сетке детектора и нормируем на единичный поток. Запросов к MAST эта ячейка не делает.


In [ ]:
print("loading/building effective PSF from HDRTAB contributors...")

import os
from collections import Counter

psf_file = Path(PSFREF if PSFREF is not None else signal_path)
psf_cache_path = out_dir / f"{stem}_psf_{PSF_SIZE}.fits"
print(f"[PSF] result cache disabled; provenance only: {psf_cache_path}")

stpsf_data_dir = Path(STPSF_DATA_DIR or os.environ.get("STPSF_PATH") or (Path.home() / "data" / "stpsf-data"))
if not stpsf_data_dir.exists():
    raise RuntimeError(f"[PSF] local STPSF data dir not found: {stpsf_data_dir}")
os.environ["STPSF_PATH"] = str(stpsf_data_dir)
print(f"[PSF] STPSF_PATH={stpsf_data_dir}")

def _text(value):
    if isinstance(value, bytes):
        return value.decode(errors="ignore").strip()
    return str(value).strip()

def _float(value, default=np.nan):
    try:
        result = float(value)
    except Exception:
        return float(default)
    return result if np.isfinite(result) else float(default)

def _row_header(row, names):
    header = fits.Header()
    for key in (
        "WCSAXES", "CRPIX1", "CRPIX2", "CRVAL1", "CRVAL2",
        "CTYPE1", "CTYPE2", "CUNIT1", "CUNIT2", "RADESYS",
        "CD1_1", "CD1_2", "CD2_1", "CD2_2",
    ):
        if key not in names:
            continue
        try:
            value = row[key]
            if np.ma.is_masked(value):
                continue
            if isinstance(value, bytes):
                value = value.decode(errors="ignore").strip()
            if hasattr(value, "item"):
                value = value.item()
            header[key] = value
        except Exception:
            continue
    return header

def _effective_filter(row, names):
    filter_name = _text(row["FILTER"]).upper() if "FILTER" in names else ""
    pupil_name = _text(row["PUPIL"]).upper() if "PUPIL" in names else ""
    if pupil_name.startswith("F") and pupil_name[-1:] in {"N", "M"}:
        return pupil_name
    return filter_name

science_center_x = float(globals().get("x0_sbf_circ", x0_center))
science_center_y = float(globals().get("y0_sbf_circ", y0_center))
science_center_ra, science_center_dec = signal_wcs.pixel_to_world_values(
    science_center_x, science_center_y
)
science_center_ra = float(science_center_ra)
science_center_dec = float(science_center_dec)
context_x = int(np.clip(round(science_center_x), 0, signal_header["NAXIS1"] - 1))
context_y = int(np.clip(round(science_center_y), 0, signal_header["NAXIS2"] - 1))

with fits.open(psf_file, memmap=True) as hdul:
    science_hdr = hdul[0].header.copy()
    hdrtab = hdul["HDRTAB"].data.copy() if "HDRTAB" in hdul else None
    context_data = np.asarray(hdul["CON"].data, dtype=np.uint32) if "CON" in hdul else None
if context_data is not None and context_data.ndim == 2:
    context_data = context_data[np.newaxis, ...]

contributors = []
if hdrtab is not None:
    names = set(hdrtab.names)
    for row_index, row in enumerate(hdrtab):
        if _effective_filter(row, names) != SIGNAL_FILTER:
            continue
        detector = _text(row["DETECTOR"]).upper() if "DETECTOR" in names else ""
        apername = _text(row["APERNAME"]).upper() if "APERNAME" in names else ""
        if not detector or not apername:
            continue
        try:
            row_wcs = WCS(_row_header(row, names)).celestial
            detector_x, detector_y = row_wcs.world_to_pixel_values(science_center_ra, science_center_dec)
            detector_x = float(detector_x)
            detector_y = float(detector_y)
        except Exception:
            continue
        subsize1 = _float(row["SUBSIZE1"], 2048.0) if "SUBSIZE1" in names else 2048.0
        subsize2 = _float(row["SUBSIZE2"], 2048.0) if "SUBSIZE2" in names else 2048.0
        inside = (
            np.isfinite(detector_x) and np.isfinite(detector_y)
            and 0.0 <= detector_x < subsize1 and 0.0 <= detector_y < subsize2
        )
        if not inside:
            continue
        context_at_center = None
        context_plane = row_index // 32
        context_bit = row_index % 32
        if context_data is not None and context_plane < context_data.shape[0]:
            context_value = int(context_data[context_plane, context_y, context_x])
            context_at_center = bool(context_value & (1 << context_bit))
        if context_at_center is False:
            continue
        contributors.append({
            "hdrtab_index": int(row_index),
            "context_plane": int(context_plane),
            "context_bit": int(context_bit),
            "context_at_center": context_at_center,
            "input_filename": _text(row["FILENAME"]) if "FILENAME" in names else "",
            "detector": detector,
            "apername": apername,
            # STPSF interprets detector_position relative to the selected
            # SIAF aperture. HDRTAB WCS already returns that coordinate.
            "detector_x": float(detector_x),
            "detector_y": float(detector_y),
            "expmid_mjd": _float(row["EXPMID"]) if "EXPMID" in names else np.nan,
            "effexptm_s": _float(row["EFFEXPTM"], 1.0) if "EFFEXPTM" in names else 1.0,
        })

detector_primary = _text(science_hdr.get("DETECTOR", "")).upper()
if detector_primary == "MULTIPLE" and not contributors:
    raise RuntimeError(
        "[PSF] DETECTOR=MULTIPLE but HDRTAB+CON did not identify a contributor at the adopted galaxy center"
    )
if not contributors:
    apername = _text(science_hdr.get("APERNAME", ""))
    detector = _text(science_hdr.get("DETECTOR", ""))
    if not apername or not detector:
        raise RuntimeError("[PSF] detector/aperture unavailable in primary header and HDRTAB")
    contributors = [{
        "hdrtab_index": -1, "context_plane": -1, "context_bit": -1,
        "context_at_center": None, "input_filename": psf_file.name,
        "detector": detector, "apername": apername,
        "detector_x": 1024.0, "detector_y": 1024.0,
        "expmid_mjd": np.nan, "effexptm_s": 1.0,
    }]

detector_weights = {}
for row in contributors:
    detector_weights[row["detector"]] = detector_weights.get(row["detector"], 0.0) + max(row["effexptm_s"], 1.0)
selected_detector = max(detector_weights, key=detector_weights.get)
selected_contributors = [row for row in contributors if row["detector"] == selected_detector]
selected_aperture = Counter(row["apername"] for row in selected_contributors).most_common(1)[0][0]
selected_contributors = [row for row in selected_contributors if row["apername"] == selected_aperture]
psf_detector_positions = [
    (int(round(row["detector_x"])), int(round(row["detector_y"])))
    for row in selected_contributors
]
print(
    f"[PSF] selected detector={selected_detector}, aperture={selected_aperture}, "
    f"contributors={len(selected_contributors)}, positions={psf_detector_positions}"
)

science_mjd_values = np.asarray([row["expmid_mjd"] for row in selected_contributors], dtype=float)
science_mjd_values = science_mjd_values[np.isfinite(science_mjd_values)]
if science_mjd_values.size:
    science_datetime = np.datetime64("1858-11-17T00:00:00") + np.timedelta64(int(round(float(np.nanmedian(science_mjd_values)) * 86400.0 * 1000.0)), "ms")
else:
    date_value = science_hdr.get("DATE-OBS", science_hdr.get("DATE"))
    time_value = science_hdr.get("TIME-OBS", "00:00:00")
    science_datetime = np.datetime64(f"{date_value}T{time_value}")

def _header_datetime(header):
    date_value = header.get("DATE-OBS", header.get("DATE"))
    if not date_value:
        return None
    time_value = header.get("TIME-OBS", "00:00:00")
    try:
        return np.datetime64(f"{date_value}T{time_value}")
    except Exception:
        return None

opd_dirs = [PSF_PROJECT_WSS_DIR, stpsf_data_dir / "MAST_JWST_WSS_OPDs"]
local_wss_files = []
for directory in opd_dirs:
    if Path(directory).exists():
        local_wss_files.extend(Path(directory).glob("*.fits"))
local_wss_files = sorted({path.resolve() for path in local_wss_files})
if not local_wss_files:
    raise RuntimeError(
        f"[PSF] no local WSS OPD files; run code/download_wss_opds.py --download (searched {opd_dirs})"
    )
opd_candidates = []
for candidate in local_wss_files:
    if candidate.stat().st_size <= 0 or candidate.stat().st_size % 2880:
        print(f"[PSF][WARN] ignoring invalid-size OPD {candidate}")
        continue
    try:
        with fits.open(candidate, memmap=False) as candidate_hdul:
            candidate_hdul.verify("exception")
            for candidate_hdu in candidate_hdul:
                if candidate_hdu.data is not None:
                    _ = candidate_hdu.data.shape
            candidate_header = candidate_hdul[0].header.copy()
    except Exception as exc:
        print(f"[PSF][WARN] ignoring unreadable/truncated OPD {candidate}: {exc}")
        continue
    candidate_datetime = _header_datetime(candidate_header)
    if candidate_datetime is None:
        continue
    signed_delta_days = float((candidate_datetime - science_datetime) / np.timedelta64(1, "D"))
    opd_candidates.append((abs(signed_delta_days), signed_delta_days, candidate.name, candidate, candidate_header))
if not opd_candidates:
    raise RuntimeError("[PSF] local WSS OPDs have no usable DATE-OBS")
opd_candidates.sort(key=lambda row: (row[0], row[2]))
opd_delta_days, opd_signed_delta_days, _, local_wss_opd, opd_hdr = opd_candidates[0]
opd_corr_id = _text(opd_hdr.get("CORR_ID", "NA"))
print(
    f"[PSF] local WSS OPD={local_wss_opd}; CORR_ID={opd_corr_id}; "
    f"signed delta={opd_signed_delta_days:+.3f} d"
)
if opd_delta_days > PSF_OPD_WARN_DELTA_DAYS:
    print(f"[PSF][WARN] OPD is {opd_delta_days:.3f} days from the science exposure")
if PSF_REQUIRE_FRESH_OPD and opd_delta_days > PSF_OPD_MAX_DELTA_DAYS:
    raise RuntimeError(
        f"[PSF] closest local OPD is stale by {opd_delta_days:.2f} d; limit={PSF_OPD_MAX_DELTA_DAYS:.2f} d. "
        "Run code/download_wss_opds.py --download before offline batch processing."
    )

sim = stpsf.instrument(science_hdr["INSTRUME"])
pupil_name = _text(science_hdr.get("PUPIL", "CLEAR")).upper()
if sim.name == "NIRCam" and pupil_name.startswith("F") and pupil_name[-1:] in {"N", "M"}:
    sim.filter = pupil_name
else:
    sim.filter = SIGNAL_FILTER
sim.set_position_from_aperture_name(selected_aperture)
if sim.name == "NIRCam" and pupil_name != "CLEAR" and not pupil_name.startswith("F") and not pupil_name.startswith("MASK"):
    sim.pupil_mask = pupil_name
# STPSF 2.1 writes the supplied filename into an ASCII-only FITS HISTORY
# card.  The project path contains Cyrillic, so pass only the ASCII basename
# while temporarily standing in the OPD directory.
previous_working_directory = Path.cwd()
try:
    os.chdir(local_wss_opd.parent)
    sim.load_wss_opd(local_wss_opd.name, verbose=True, plot=False)
finally:
    os.chdir(previous_working_directory)
sim.options["output_mode"] = "both"

def _psf_array_from_hdu(hdu):
    array = np.asarray(hdu.data, dtype=float)
    if array.ndim == 3:
        array = array.sum(axis=0)
    return array

def _select_psf_hdu(hdul):
    if isinstance(hdul, fits.HDUList):
        # Prefer the distorted oversampled image, then explicitly shrink it
        # onto the science mosaic scale. DET_DIST is only a fallback.
        for extname in ("OVERDIST", "DET_DIST", "DET_SAMP", "OVERSAMP"):
            try:
                hdu = hdul[extname]
                return extname, _psf_array_from_hdu(hdu), hdu.header.copy()
            except Exception:
                pass
        for index, hdu in enumerate(hdul):
            if getattr(hdu, "data", None) is not None:
                return hdu.header.get("EXTNAME", f"EXT{index}"), _psf_array_from_hdu(hdu), hdu.header.copy()
    if isinstance(hdul, fits.PrimaryHDU):
        return "PRIMARY", _psf_array_from_hdu(hdul), hdul.header.copy()
    return "ARRAY", np.asarray(hdul, dtype=float), fits.Header()

def _center_crop_or_pad(array, size):
    output = np.zeros((size, size), dtype=float)
    source_y0 = max(0, (array.shape[0] - size) // 2)
    source_x0 = max(0, (array.shape[1] - size) // 2)
    source_y1 = min(array.shape[0], source_y0 + size)
    source_x1 = min(array.shape[1], source_x0 + size)
    height = source_y1 - source_y0
    width = source_x1 - source_x0
    target_y0 = (size - height) // 2
    target_x0 = (size - width) // 2
    output[target_y0:target_y0 + height, target_x0:target_x0 + width] = array[source_y0:source_y1, source_x0:source_x1]
    return output

science_pixel_scale_arcsec = float(np.sqrt(pix_area))
psf_arrays = []
psf_weights = []
psf_contributor_rows = []
psf_selected_ext = None
psf_native_pixel_scale_arcsec = np.nan
psf_native_detector_scale_arcsec = np.nan
psf_resample_factor = np.nan
for contributor, detector_position in zip(selected_contributors, psf_detector_positions):
    sim.detector_position = detector_position
    psf_hdul = sim.calc_psf(
        nlambda=PSF_NLAMBDA,
        fov_pixels=PSF_SIZE,
        fft_oversample=4,
        detector_oversample=PSF_DETECTOR_OVERSAMPLE,
        add_distortion=True,
    )
    selected_ext, native_psf, selected_header = _select_psf_hdu(psf_hdul)
    native_psf = np.nan_to_num(native_psf, nan=0.0)
    native_sum = float(native_psf.sum())
    if not np.isfinite(native_sum) or native_sum <= 0.0:
        raise RuntimeError(f"[PSF] invalid native PSF sum={native_sum} at {detector_position}")
    native_psf /= native_sum
    native_scale = _float(selected_header.get("PIXELSCL"))
    if not np.isfinite(native_scale):
        raise RuntimeError(f"[PSF] {selected_ext} has no PIXELSCL")
    native_oversample = int(round(_float(selected_header.get("OVERSAMP"), 1.0)))
    native_detector_scale = (
        native_scale * max(native_oversample, 1)
        if selected_ext in {"OVERDIST", "OVERSAMP"}
        else native_scale
    )
    zoom_factor = float(native_scale / science_pixel_scale_arcsec)
    resampled = ndimage.zoom(
        native_psf,
        zoom=zoom_factor,
        order=PSF_RESAMPLE_ORDER,
        mode="constant",
        cval=0.0,
        prefilter=True,
    )
    resampled = _center_crop_or_pad(resampled, PSF_SIZE)
    resampled_sum = float(resampled.sum())
    if not np.isfinite(resampled_sum) or resampled_sum <= 0.0:
        raise RuntimeError(f"[PSF] invalid resampled PSF sum={resampled_sum} at {detector_position}")
    resampled /= resampled_sum
    weight = max(float(contributor["effexptm_s"]), 1.0)
    psf_arrays.append(resampled)
    psf_weights.append(weight)
    psf_selected_ext = selected_ext
    psf_native_pixel_scale_arcsec = native_scale
    psf_native_detector_scale_arcsec = native_detector_scale
    psf_resample_factor = zoom_factor
    psf_contributor_rows.append({
        **contributor,
        "normalized_weight": np.nan,
        "native_psf_pixel_scale_arcsec": native_scale,
        "native_detector_pixel_scale_arcsec": native_detector_scale,
        "science_pixel_scale_arcsec": science_pixel_scale_arcsec,
        "resample_factor": zoom_factor,
        "resample_method": f"scipy_ndimage_zoom_order{PSF_RESAMPLE_ORDER}_center_crop_pad",
        "psf_extension": selected_ext,
        "detector_oversample": PSF_DETECTOR_OVERSAMPLE,
        "native_output_oversample": native_oversample,
        "opd_file": local_wss_opd.name,
        "opd_corr_id": opd_corr_id,
        "opd_signed_delta_days": opd_signed_delta_days,
        "method_id": "stpsf_hdrtab_context_v1",
        "method_variant": "center_context_exposure_weighted_scale_resampled",
        "quantity_family": "effective_psf_contributor",
        "quantity_name": "psf_weight",
        "primary_unit": "dimensionless",
        "method_limitations": "center contributors only; no explicit drizzle kernel or ring-position weighting",
        "siaf_prd_version": LOCAL_SIAF_PRD_VERSION,
    })

psf_weights = np.asarray(psf_weights, dtype=float)
psf_weights /= psf_weights.sum()
for row, weight in zip(psf_contributor_rows, psf_weights):
    row["normalized_weight"] = float(weight)
psf = np.average(np.stack(psf_arrays, axis=0), axis=0, weights=psf_weights)
psf = np.nan_to_num(psf, nan=0.0)
psf /= float(psf.sum())
psf_pixel_scale_arcsec = science_pixel_scale_arcsec
psf_scale_rel_error = 0.0
psf_native_scale_rel_error = abs(
    psf_native_detector_scale_arcsec / science_pixel_scale_arcsec - 1.0
)
psf_method_id = "stpsf_center_context_exposure_weighted_scale_resampled_v1"
psf_method_limitations = "center contributors only; scale matched; no explicit drizzle kernel or ring-position weighting"
psf_detector_set = selected_detector
psf_input_count = len(psf_contributor_rows)
save_stage_table("10_psf", "contributors", pd.DataFrame(psf_contributor_rows))

output_header["SBFSPX"] = (science_pixel_scale_arcsec, "science arcsec per pixel")
output_header["SBFPSCL"] = (psf_pixel_scale_arcsec, "effective PSF arcsec per pixel")
output_header["SBFPSFD"] = (selected_detector, "effective PSF detector")
output_header["SBFPSFN"] = (psf_input_count, "effective PSF contributors")
output_header["SBFPSFM"] = (psf_method_id[:68], "effective PSF method")
output_header["SBFSIAF"] = (LOCAL_SIAF_PRD_VERSION[:68], "local SIAF PRD version")
output_header["SBFOPDD"] = (float(opd_signed_delta_days), "OPD minus science date, days")
output_header["SBFOPDC"] = (opd_corr_id[:68], "WSS OPD correction id")
print(
    f"[PSF] method={psf_method_id}; shape={psf.shape}; sum={psf.sum():.6f}; "
    f"native-scale mismatch={psf_native_scale_rel_error:.3%}; effective scale={psf_pixel_scale_arcsec:.8f} arcsec/pix"
)


In [ ]:
def measure_sbf_spectral_region(
    region_mask,
    resid,
    model,
    premask,
    psf,
    pix_area,
    kmin=None,
    kmax=None,
    fft_workers=None,
    n_e_realizations=None,
    kbins_n=None,
    region_name="region",
    region_role="science",
    region_origin_x=0,
    region_origin_y=0,
):
    if kmin is None or kmax is None:
        kmin, kmax = FFT_K_RANGE_MAIN
    if fft_workers is None:
        fft_workers = FFT_WORKERS
    if n_e_realizations is None:
        n_e_realizations = FFT_E_REALIZATIONS_DIAG
    if kbins_n is None:
        kbins_n = FFT_KBINS_N

    mask_sbf_local = premask | (~region_mask)
    window = (~mask_sbf_local) & np.isfinite(resid) & np.isfinite(model) & (model > 0.0)

    n_use = int(window.sum())
    if n_use < MIN_PIXELS_SBF:
        return None

    Imean = float(np.nanmean(model[window]))
    if (not np.isfinite(Imean)) or (Imean <= 0.0):
        return None

    data_fft = np.zeros_like(resid, dtype=float)
    data_fft[window] = resid[window]
    data_fft[window] -= float(np.nanmean(data_fft[window]))

    with set_workers(fft_workers):
        F = fft2(data_fft)

    P2d = (np.abs(F) ** 2) / float(n_use)

    ny, nx = resid.shape
    ky = fftshift(fftfreq(ny))
    kx = fftshift(fftfreq(nx))
    KX, KY = np.meshgrid(kx, ky)
    kr = np.hypot(KX, KY)

    P2d_s = fftshift(P2d)
    kbins = np.linspace(0.0, float(kr.max()), kbins_n)

    Pk_vals = np.full(len(kbins) - 1, np.nan, dtype=float)
    Pk_k = np.full_like(Pk_vals, np.nan)
    for i in range(len(Pk_vals)):
        sel = (kr >= kbins[i]) & (kr < kbins[i + 1])
        vals = P2d_s[sel]
        if vals.size >= MIN_POINTS_PK_BIN:
            Pk_vals[i] = float(np.nanmedian(vals))
            Pk_k[i] = 0.5 * (kbins[i] + kbins[i + 1])

    mP = np.isfinite(Pk_vals) & np.isfinite(Pk_k) & (Pk_k > 0.0)
    if int(mP.sum()) < MIN_POINTS_FIT:
        return None

    kP = Pk_k[mP]
    P = Pk_vals[mP]

    big_psf = np.zeros((ny, nx), dtype=float)
    py, px = psf.shape
    y0_psf = ny // 2 - py // 2
    x0_psf = nx // 2 - px // 2
    big_psf[y0_psf:y0_psf + py, x0_psf:x0_psf + px] = psf

    with set_workers(fft_workers):
        F_psf = fft2(big_psf)

    rng = np.random.default_rng(FFT_RNG_SEED)
    Ek_stack = []
    for _ in range(n_e_realizations):
        noise = rng.normal(loc=0.0, scale=1.0, size=(ny, nx))
        with set_workers(fft_workers):
            F_noise = fft2(noise)
            sim = np.real(np.fft.ifft2(F_noise * F_psf))

        sim_masked = np.zeros_like(sim, dtype=float)
        sim_masked[window] = sim[window]
        sim_masked[window] -= float(np.nanmean(sim_masked[window]))

        with set_workers(fft_workers):
            F_sim = fft2(sim_masked)

        E2d_sim = (np.abs(F_sim) ** 2) / float(n_use)
        E2d_sim_s = fftshift(E2d_sim)

        Ek_vals_i = np.full(len(kbins) - 1, np.nan, dtype=float)
        for j in range(len(Ek_vals_i)):
            sel = (kr >= kbins[j]) & (kr < kbins[j + 1])
            vals = E2d_sim_s[sel]
            if vals.size >= MIN_POINTS_PK_BIN:
                Ek_vals_i[j] = float(np.nanmedian(vals))
        Ek_stack.append(Ek_vals_i)

    Ek_stack = np.array(Ek_stack, dtype=float)
    Ek_vals = np.nanmedian(Ek_stack, axis=0)
    Ek_k = 0.5 * (kbins[:-1] + kbins[1:])

    mE = np.isfinite(Ek_vals) & np.isfinite(Ek_k) & (Ek_k > 0.0)
    if int(mE.sum()) < MIN_POINTS_FIT:
        return None

    kE = Ek_k[mE]
    E = Ek_vals[mE]

    sel = (kP >= kmin) & (kP <= kmax)
    if int(sel.sum()) < MIN_POINTS_FIT:
        return None

    E_int = np.interp(kP[sel], kE, E, left=np.nan, right=np.nan)
    x = E_int
    y = P[sel]
    good = np.isfinite(x) & np.isfinite(y) & (x > 0.0) & (y > 0.0)
    x = x[good]
    y = y[good]
    if x.size < MIN_POINTS_FIT:
        return None

    corr = float(np.corrcoef(x, y)[0, 1])
    A = np.vstack([x, np.ones_like(x)]).T
    P0, P1 = np.linalg.lstsq(A, y, rcond=None)[0]
    if (not np.isfinite(P0)) or (P0 <= 0.0):
        return None

    y_fit = P0 * x + P1
    fit_resid = y - y_fit
    fit_resid_std = float(np.nanstd(fit_resid, ddof=1)) if x.size > 2 else float(np.nanstd(fit_resid))

    P0_fit_sigma = np.nan
    mbar_fit_sigma_raw = np.nan
    dof = max(int(x.size) - 2, 1)
    try:
        sigma2_fit = float(np.nansum(fit_resid * fit_resid) / dof)
        cov_fit = sigma2_fit * np.linalg.pinv(A.T @ A)
        P0_fit_sigma = float(np.sqrt(max(cov_fit[0, 0], 0.0)))
        if np.isfinite(P0_fit_sigma) and P0_fit_sigma > 0.0:
            mbar_fit_sigma_raw = float((2.5 / np.log(10.0)) * P0_fit_sigma / P0)
    except Exception:
        pass

    den = P0 + P1
    frac = float(P0 / den) if np.isfinite(den) and den != 0.0 else np.nan

    pr_info = estimate_pr_for_region(
        region_mask=region_mask,
        model=model,
        region_name=region_name,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )
    power_info = solve_sbf_power_budget(P0=P0, Imean=Imean, Pr=pr_info.get("Pr", np.nan), pix_area=pix_area, P0_fit_sigma=P0_fit_sigma)

    return {
        "measurement_ok": bool(power_info.get("measurement_ok", False)),
        "failure_reason": power_info.get("failure_reason", ""),
        "region_name": region_name,
        "region_role": region_role,
        "n_use": n_use,
        "Imean": Imean,
        "P0": float(P0),
        "P1": float(P1),
        "P_fluc": power_info.get("P_fluc", np.nan),
        "Pr": pr_info.get("Pr", np.nan),
        "Pr_over_P0": power_info.get("Pr_over_P0", np.nan),
        "frac": frac,
        "corr": corr,
        "n_fit": int(x.size),
        "fit_resid_std": fit_resid_std,
        "P0_fit_sigma": P0_fit_sigma,
        "mbar_fit_sigma_raw": mbar_fit_sigma_raw,
        "mbar_fit_sigma": power_info.get("mbar_fit_sigma", np.nan),
        "Pf_spec_raw": power_info.get("Pf_spec_raw", np.nan),
        "Pf_spec": power_info.get("Pf_spec", np.nan),
        "Pf_spec_sigma_formal_raw": power_info.get("Pf_spec_sigma_formal_raw", np.nan),
        "Pf_spec_sigma_formal": power_info.get("Pf_spec_sigma_formal", np.nan),
        "mbar_spec_raw": power_info.get("mbar_spec_raw", np.nan),
        "mbar_spec": power_info.get("mbar_spec", np.nan),
        "n_detected_sources": pr_info.get("n_detected_sources", 0),
        "n_detected_sources_total": pr_info.get("n_detected_sources_total", 0),
        "m_lim": pr_info.get("m_lim", np.nan),
        "m_lim_method": pr_info.get("m_lim_method", "unknown"),
        "lf_model": pr_info.get("lf_model", "combined_power_law"),
        "lf_gamma": pr_info.get("lf_gamma", np.nan),
        "lf_phi_mlim": pr_info.get("lf_phi_mlim", np.nan),
        "lf_fit_method": pr_info.get("lf_fit_method", "unknown"),
        "lf_fit_rmse": pr_info.get("lf_fit_rmse", np.nan),
        "lf_n_fit_bins": pr_info.get("lf_n_fit_bins", 0),
        "lf_n_fit_sources": pr_info.get("lf_n_fit_sources", 0),
        "lf_fit_mbright": pr_info.get("lf_fit_mbright", np.nan),
        "lf_fit_mfaint": pr_info.get("lf_fit_mfaint", np.nan),
        "pr_note": pr_info.get("pr_note", ""),
    }

def measure_sbf_spec_for_row(
    row,
    resid,
    model,
    premask,
    psf,
    pix_area,
    kmin=None,
    kmax=None,
    fft_workers=None,
    n_e_realizations=None,
    kbins_n=None,
):
    """
    Считает spectral SBF для одного isophote-window row.

    Важно: raw P0 сам по себе ещё не является stellar SBF signal, потому что
    unresolved contaminants тоже дают PSF-shaped power. Поэтому итогом считаем
    corrected power P_fluc = P0 - Pr.
    """
    ny_r, nx_r = resid.shape
    yy_r, xx_r = np.indices((ny_r, nx_r), dtype=float)

    sma_in = float(row["sma_in"])
    sma_out = float(row["sma_out"])
    x0_ann = float(row["x0"])
    y0_ann = float(row["y0"])
    pa_ann = float(row["pa"])
    q_ann = max(GEOM_Q_FLOOR, float(row["q"]))

    dx = xx_r - x0_ann
    dy = yy_r - y0_ann
    cosp = np.cos(pa_ann)
    sinp = np.sin(pa_ann)
    xp = dx * cosp + dy * sinp
    yp = -dx * sinp + dy * cosp
    r_ell = np.sqrt(xp * xp + (yp / q_ann) * (yp / q_ann))
    annulus_ell = (r_ell >= sma_in) & (r_ell <= sma_out)

    out = measure_sbf_spectral_region(
        region_mask=annulus_ell,
        resid=resid,
        model=model,
        premask=premask,
        psf=psf,
        pix_area=pix_area,
        kmin=kmin,
        kmax=kmax,
        fft_workers=fft_workers,
        n_e_realizations=n_e_realizations,
        kbins_n=kbins_n,
        region_name=f"plateau_{sma_in:.1f}_{sma_out:.1f}",
        region_role="diagnostic",
        region_origin_x=0,
        region_origin_y=0,
    )
    if out is None:
        return None

    out.update({
        "sma_in": sma_in,
        "sma_out": sma_out,
        "sma_mid": float(row["sma_mid"]),
        "i0": int(row["i0"]),
        "i1": int(row["i1"]),
    })
    return out


## Измерение SBF в двух круговых кольцах

Измеряем внутреннее кольцо 8.2–16.4 угл. сек. и внешнее 16.4–32.8 угл. сек. Основное окно пространственных частот задаётся `FFT_K_RANGE_MAIN`; ещё два окна служат проверкой устойчивости.


In [ ]:
def build_region_mask_ellipse(shape, x0, y0, q, pa, sma_in, sma_out):
    ny, nx = shape
    yy, xx = np.ogrid[:ny, :nx]

    dx = xx.astype(float) - float(x0)
    dy = yy.astype(float) - float(y0)

    cosp = np.cos(pa)
    sinp = np.sin(pa)

    xp = dx * cosp + dy * sinp
    yp = -dx * sinp + dy * cosp

    r_ell = np.sqrt(xp * xp + (yp / q) * (yp / q))
    return (r_ell >= sma_in) & (r_ell <= sma_out)

def build_region_mask_circle(shape, x0, y0, r_in, r_out):
    ny, nx = shape
    yy, xx = np.ogrid[:ny, :nx]
    rr2 = (xx.astype(float) - float(x0)) ** 2 + (yy.astype(float) - float(y0)) ** 2
    return (rr2 >= float(r_in) ** 2) & (rr2 <= float(r_out) ** 2)


In [ ]:
def measure_sbf_for_mask(
    region_mask,
    resid,
    model,
    premask,
    psf,
    pix_area,
    kmin,
    kmax,
    fft_workers=None,
    n_e_realizations=None,
    kbins_n=None,
    region_name="region",
    region_role="science",
    region_origin_x=0,
    region_origin_y=0,
):
    return measure_sbf_spectral_region(
        region_mask=region_mask,
        resid=resid,
        model=model,
        premask=premask,
        psf=psf,
        pix_area=pix_area,
        kmin=kmin,
        kmax=kmax,
        fft_workers=fft_workers,
        n_e_realizations=n_e_realizations,
        kbins_n=kbins_n,
        region_name=region_name,
        region_role=region_role,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )


In [ ]:
print("measuring SBF in Jensen/TRGB-SBF circular annuli...")

if "science_model" not in globals() or "science_resid" not in globals():
    raise RuntimeError("[КОЛЬЦА SBF] unified science path is unavailable; rerun the science-path cell")

pix_scale = float(np.sqrt(pix_area))
r1_in, r1_out = SBF_LIT_INNER_ARCSEC[0] / pix_scale, SBF_LIT_INNER_ARCSEC[1] / pix_scale
r2_in, r2_out = SBF_LIT_OUTER_ARCSEC[0] / pix_scale, SBF_LIT_OUTER_ARCSEC[1] / pix_scale

print(
    f"[КОЛЬЦА SBF] unified science path: residual={science_resid_name}, model={science_model_name}; "
    "fixed circular annuli are the science regions"
)
print(
    f"[КОЛЬЦА SBF] circular center = ({x0_sbf_circ:.2f}, {y0_sbf_circ:.2f}), "
    f"radii = {r1_in:.1f}-{r1_out:.1f} px and {r2_in:.1f}-{r2_out:.1f} px"
)

regions = [
    {
        "region": "circular_inner_lit",
        "role": "science",
        "shape": "circle",
        "mask": build_region_mask_circle(
            science_resid.shape, x0_sbf_circ, y0_sbf_circ, r1_in, r1_out
        ),
        "rin_px": r1_in,
        "rout_px": r1_out,
        "rin_arcsec": SBF_LIT_INNER_ARCSEC[0],
        "rout_arcsec": SBF_LIT_INNER_ARCSEC[1],
        "resid_data": science_resid,
        "model_data": science_model,
        "resid_source": science_resid_name,
        "model_source": science_model_name,
        "origin_x": 0,
        "origin_y": 0,
    },
    {
        "region": "circular_outer_lit",
        "role": "science",
        "shape": "circle",
        "mask": build_region_mask_circle(
            science_resid.shape, x0_sbf_circ, y0_sbf_circ, r2_in, r2_out
        ),
        "rin_px": r2_in,
        "rout_px": r2_out,
        "rin_arcsec": SBF_LIT_OUTER_ARCSEC[0],
        "rout_arcsec": SBF_LIT_OUTER_ARCSEC[1],
        "resid_data": science_resid,
        "model_data": science_model,
        "resid_source": science_resid_name,
        "model_source": science_model_name,
        "origin_x": 0,
        "origin_y": 0,
    },
]

k_windows = list(SBF_REGION_K_WINDOWS)
rows_df = []
sbf_usable_mask_by_region = {}

for reg in regions:
    region_mask = reg["mask"]
    reg_resid = reg["resid_data"]
    reg_model = reg["model_data"]
    region_pixels = int(region_mask.sum())
    usable_region = region_mask & (~premask) & np.isfinite(reg_resid) & np.isfinite(reg_model) & (reg_model > 0.0)
    sbf_usable_mask_by_region[reg["region"]] = usable_region.copy()
    usable_pixels = int(usable_region.sum())
    usable_fraction = float(usable_pixels / region_pixels) if region_pixels > 0 else np.nan

    print(
        f"[КОЛЬЦА SBF] {reg['region']} ({reg['role']}): "
        f"N_region={region_pixels}, N_usable={usable_pixels}, "
        f"usable_fraction={usable_fraction:.3f}, residual={reg['resid_source']}, model={reg['model_source']}"
    )
    src_reg_summary = summarize_sources_in_region_mask(
        region_mask,
        region_origin_x=reg['origin_x'],
        region_origin_y=reg['origin_y'],
    )
    print(
        f"[SRC-CAT-REGION] {reg['region']}: overlap={src_reg_summary['n_overlap']}, "
        f"valid={src_reg_summary['n_valid']}, nonpositive_flux={src_reg_summary['n_nonpositive_flux']}, "
        f"mag_med={src_reg_summary['mag_med']:.3f}"
    )

    for kmin, kmax in k_windows:
        out = measure_sbf_for_mask(
            region_mask=region_mask,
            resid=reg_resid,
            model=reg_model,
            premask=premask,
            psf=psf,
            pix_area=pix_area,
            kmin=kmin,
            kmax=kmax,
            fft_workers=FFT_WORKERS,
            n_e_realizations=FFT_E_REALIZATIONS_MAIN,
            kbins_n=FFT_KBINS_N,
            region_name=reg["region"],
            region_role=reg["role"],
            region_origin_x=reg["origin_x"],
            region_origin_y=reg["origin_y"],
        )

        base_row = {
            "region": reg["region"],
            "role": reg["role"],
            "shape": reg["shape"],
            "rin_px": reg["rin_px"],
            "rout_px": reg["rout_px"],
            "rin_arcsec": reg["rin_arcsec"],
            "rout_arcsec": reg["rout_arcsec"],
            "resid_source": reg["resid_source"],
            "model_source": reg["model_source"],
            "region_pixels": region_pixels,
            "usable_pixels": usable_pixels,
            "usable_fraction": usable_fraction,
            "kmin": kmin,
            "kmax": kmax,
            "schema_version": TABLE_SCHEMA_VERSION,
            "pipeline_version": PIPELINE_VERSION,
            "target_id": str(TARGET_GALAXY),
            "program_id": str(signal_primary_header.get("PROGRAM", "")).lstrip("0") or "0",
            "observation_id": str(signal_primary_header.get("OBSERVTN", "")),
            "signal_filter": SIGNAL_FILTER,
            "color_filter": COLOR_FILTER or "NONE",
            "record_type": "sbf_fit",
            "quantity_family": "sbf_power_and_apparent_magnitude",
            "quantity_name": "mbar_spec",
            "primary_unit": "AB mag",
            "method_id": "fft_psf_window_pr_corrected_v1",
            "method_variant": f"k={kmin:.3f}..{kmax:.3f}",
            "region_id": reg["region"],
            "region_role": reg["role"],
            "region_geometry": "fixed_circular_annulus",
            "ring_selection": "fixed_literature_qc_single_annulus",
            "power_correction": "P_fluc=P0-Pr",
            "calibration_applied": False,
            "distance_computed": False,
            "psf_method": psf_method_id,
            "psf_detector_set": psf_detector_set,
            "psf_opd_corr_id": opd_corr_id,
            "psf_opd_delta_days": float(opd_signed_delta_days),
            "psf_input_count": int(psf_input_count),
            "psf_extension": psf_selected_ext,
            "psf_native_scale_rel_error": float(psf_native_scale_rel_error),
            "psf_method_limitations": psf_method_limitations,
            "psf_siaf_prd_version": LOCAL_SIAF_PRD_VERSION,
        }

        if out is None:
            rows_df.append({**base_row, "status": "failed_raw"})
            print(f"[КОЛЬЦА SBF]   k={kmin:.3f}..{kmax:.3f} -> raw spectral fit failed")
            continue

        status = "ok" if out.get("measurement_ok", False) else "invalid_corrected"
        rows_df.append({**base_row, "status": status, **out})

        if status == "ok":
            corr_text = f"{out['Pr_over_P0']:.3%}" if np.isfinite(out.get("Pr_over_P0", np.nan)) else "n/a"
            print(
                f"[КОЛЬЦА SBF]   k={kmin:.3f}..{kmax:.3f}: raw mbar={out['mbar_spec_raw']:.4f}, "
                f"corrected mbar={out['mbar_spec']:.4f}, P0={out['P0']:.3e}, Pr={out['Pr']:.3e}, "
                f"P_fluc={out['P_fluc']:.3e}, Pr/P0={corr_text}, "
                f"Nsrc_valid={out['n_detected_sources']}, Nsrc_overlap={out.get('n_detected_sources_total', 0)}, "
                f"m_lim={out['m_lim']:.3f}, gamma={out['lf_gamma']:.3f}"
            )
        else:
            print(
                f"[КОЛЬЦА SBF]   k={kmin:.3f}..{kmax:.3f}: raw mbar={out['mbar_spec_raw']:.4f}, "
                f"corrected measurement invalid ({out.get('failure_reason', '')}), "
                f"P0={out['P0']:.3e}, Pr={out['Pr']:.3e}, P_fluc={out['P_fluc']:.3e}"
            )

df_sbf = pd.DataFrame(rows_df)

main_kmin, main_kmax = FFT_K_RANGE_MAIN
reference_kmin = 0.03
ring_qc = {}
for region_name in [reg["region"] for reg in regions]:
    group = df_sbf[df_sbf["region"].eq(region_name)].copy()
    main_rows = group[
        np.isclose(group["kmin"].astype(float), float(main_kmin))
        & np.isclose(group["kmax"].astype(float), float(main_kmax))
    ]
    reference_rows = group[
        np.isclose(group["kmin"].astype(float), float(reference_kmin))
        & np.isclose(group["kmax"].astype(float), float(main_kmax))
    ]
    failures = []
    warnings_qc = []
    main_row = None if main_rows.empty else main_rows.iloc[0]
    reference_row = None if reference_rows.empty else reference_rows.iloc[0]
    if main_row is None:
        failures.append("main k-window missing")
        k_shift = np.nan
        usable_fraction_qc = np.nan
        pr_fraction = np.nan
        corr_value = np.nan
    else:
        usable_fraction_qc = float(main_row.get("usable_fraction", np.nan))
        pr_fraction = float(main_row.get("Pr_over_P0", np.nan))
        corr_value = float(main_row.get("corr", np.nan))
        measurement_ok = bool(main_row.get("measurement_ok", False)) and main_row.get("status") == "ok"
        mbar_value = float(main_row.get("mbar_spec", np.nan))
        p_fluc_value = float(main_row.get("P_fluc", np.nan))
        n_use_value = float(main_row.get("n_use", 0))
        if not measurement_ok:
            failures.append("main measurement invalid")
        if not np.isfinite(mbar_value):
            failures.append("mbar nonfinite")
        if not np.isfinite(p_fluc_value) or p_fluc_value <= 0.0:
            failures.append("P_fluc nonpositive")
        if n_use_value < MIN_PIXELS_SBF:
            failures.append(f"n_use<{MIN_PIXELS_SBF}")
        if not np.isfinite(usable_fraction_qc) or usable_fraction_qc < SBF_QC_MIN_USABLE_FRACTION:
            failures.append(f"usable_fraction<{SBF_QC_MIN_USABLE_FRACTION:.2f}")
        elif usable_fraction_qc < SBF_QC_WARN_USABLE_FRACTION:
            warnings_qc.append(f"usable_fraction<{SBF_QC_WARN_USABLE_FRACTION:.2f}")
        if not np.isfinite(pr_fraction):
            warnings_qc.append("Pr/P0 nonfinite")
        elif pr_fraction > SBF_QC_MAX_PR_OVER_P0:
            warnings_qc.append(f"Pr/P0>{SBF_QC_MAX_PR_OVER_P0:.2f}")
        if not np.isfinite(corr_value):
            warnings_qc.append("corr nonfinite")
        elif corr_value < SBF_QC_MIN_CORR:
            warnings_qc.append(f"corr<{SBF_QC_MIN_CORR:.2f}")
        if reference_row is not None and np.isfinite(reference_row.get("mbar_spec", np.nan)) and np.isfinite(mbar_value):
            k_shift = float(abs(mbar_value - float(reference_row["mbar_spec"])))
            if k_shift > SBF_QC_MAX_KSHIFT_MAG:
                warnings_qc.append(f"k03-k04 shift>{SBF_QC_MAX_KSHIFT_MAG:.2f} mag")
        else:
            k_shift = np.nan
            warnings_qc.append("k03 reference unavailable")

    qc_status = "FAIL" if failures else ("WARN" if warnings_qc else "PASS")
    reasons = failures + warnings_qc
    ring_qc[region_name] = {
        "status": qc_status,
        "reasons": "; ".join(reasons) if reasons else "all encoded checks passed",
        "k_stability_mag": k_shift,
        "usable_fraction": usable_fraction_qc,
        "main_row": main_row,
    }

eligible_regions = [
    region_name for region_name, info in ring_qc.items()
    if info["status"] != "FAIL" and info["main_row"] is not None
]
qc_rank = {"PASS": 0, "WARN": 1, "FAIL": 2}
region_preference = {"circular_inner_lit": 0, "circular_outer_lit": 1}
eligible_regions.sort(
    key=lambda region_name: (
        qc_rank[ring_qc[region_name]["status"]],
        ring_qc[region_name]["k_stability_mag"] if np.isfinite(ring_qc[region_name]["k_stability_mag"]) else np.inf,
        -ring_qc[region_name]["usable_fraction"] if np.isfinite(ring_qc[region_name]["usable_fraction"]) else np.inf,
        region_preference.get(region_name, 99),
    )
)
selected_sbf_region = eligible_regions[0] if eligible_regions else None
selected_sbf_selection_method = "single_annulus_qc_rank_then_k03_k04_stability_v1"

df_sbf["ring_qc_status"] = df_sbf["region"].map(lambda value: ring_qc.get(value, {}).get("status", "FAIL"))
df_sbf["ring_qc_reasons"] = df_sbf["region"].map(lambda value: ring_qc.get(value, {}).get("reasons", "missing QC"))
df_sbf["ring_k_stability_mag"] = df_sbf["region"].map(lambda value: ring_qc.get(value, {}).get("k_stability_mag", np.nan))
df_sbf["selected_for_final"] = (
    df_sbf["region"].eq(selected_sbf_region)
    & np.isclose(df_sbf["kmin"].astype(float), float(main_kmin))
    & np.isclose(df_sbf["kmax"].astype(float), float(main_kmax))
)
df_sbf["selection_reason"] = np.where(
    df_sbf["selected_for_final"],
    selected_sbf_selection_method,
    "diagnostic ring or diagnostic k-window",
)
print(f"[КОЛЬЦА SBF] selected region={selected_sbf_region}; method={selected_sbf_selection_method}")
for region_name, info in ring_qc.items():
    print(
        f"[КОЛЬЦА SBF][QC] {region_name}: {info['status']}; "
        f"k03-k04={info['k_stability_mag']}; {info['reasons']}"
    )
display(df_sbf)


## Рабочие остатки в двух кольцах

Одним пятым FITS сохраняем объединение двух колец. Вне реально используемых пикселей записывается `NaN`.


In [ ]:
print("Сохраняю рабочие остатки в двух измерительных кольцах...")

if not all(name in globals() for name in ("science_resid", "science_model", "premask")):
    raise RuntimeError("Рабочие остатки, модель или маска ещё не построены")

маска_двух_колец = regions[0]["mask"] | regions[1]["mask"]
рабочие_пиксели_колец = (
    маска_двух_колец
    & (~premask)
    & np.isfinite(science_resid)
    & np.isfinite(science_model)
    & (science_model > 0.0)
)
остатки_в_двух_кольцах = np.full(science_resid.shape, np.nan, dtype=np.float32)
остатки_в_двух_кольцах[рабочие_пиксели_колец] = science_resid[рабочие_пиксели_колец]

путь_колец = out_dir / f"{stem}_05_остатки_общие_рабочие_два_кольца.fits"
сохранить_скалярный_fits(
    путь_колец, остатки_в_двух_кольцах, output_header,
    "Working residual restricted to the two SBF measurement annuli",
)
print(f"[FITS] 05_остатки_общие_рабочие_два_кольца: {рабочие_пиксели_колец.sum()} пикс. -> {путь_колец}")


## Исходная и исправленная амплитуда SBF

Таблица показывает исходную мощность $P_0$, поправку $P_r$, исправленную мощность и соответствующие видимые звёздные величины для каждого кольца и частотного окна.


In [ ]:
compare_columns = [
    "target_id", "program_id", "observation_id", "signal_filter", "color_filter",
    "method_id", "method_variant", "quantity_family", "quantity_name", "primary_unit",
    "region", "role", "status", "ring_qc_status", "ring_qc_reasons",
    "ring_k_stability_mag", "selected_for_final", "selection_reason",
    "rin_arcsec", "rout_arcsec", "usable_fraction", "kmin", "kmax",
    "mbar_spec_raw", "mbar_spec", "P0", "Pr", "P_fluc", "Pr_over_P0",
    "Imean", "n_detected_sources", "m_lim", "lf_gamma", "lf_fit_method",
    "psf_method", "psf_detector_set", "psf_opd_corr_id", "psf_opd_delta_days", "psf_input_count",
    "psf_extension", "psf_native_scale_rel_error", "psf_method_limitations", "psf_siaf_prd_version",
    "calibration_applied", "distance_computed",
]
compare_columns = [column for column in compare_columns if column in df_sbf.columns]
df_sbf_compare = df_sbf[compare_columns].copy()
display(df_sbf_compare.sort_values(["region", "kmin", "kmax"]).reset_index(drop=True))
save_stage_table("11_sbf", "all_measurements", df_sbf)
save_stage_table("11_sbf", "comparison", df_sbf_compare)


## Итог по двум кольцам

При успешном измерении обоих колец объединяем их с весами, обратными квадратам ошибок. Итоговая ошибка не может быть меньше половины расхождения между кольцами.


In [ ]:
print("selecting one fixed circular SBF annulus by explicit QC...")

def _finite_positive(value):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return False
    return np.isfinite(value) and value > 0.0

def _robust_mag_scatter(values):
    array = np.asarray(values, dtype=float)
    array = array[np.isfinite(array)]
    if array.size >= 3:
        median = float(np.nanmedian(array))
        mad_sigma = float(1.4826 * np.nanmedian(np.abs(array - median)))
        std_sigma = float(np.nanstd(array, ddof=1))
        if _finite_positive(mad_sigma):
            return mad_sigma, "k-window MAD"
        if _finite_positive(std_sigma):
            return std_sigma, "k-window std"
    if array.size >= 2:
        std_sigma = float(np.nanstd(array, ddof=1))
        half_range = float(0.5 * (np.nanmax(array) - np.nanmin(array)))
        if _finite_positive(std_sigma):
            return std_sigma, "k-window std"
        if _finite_positive(half_range):
            return half_range, "k-window half-range"
    return np.nan, "no k-window scatter"

df_sbf_ok = df_sbf[df_sbf["status"].eq("ok")].copy()
region_sigma_info = {}
for region_name, group in df_sbf_ok.groupby("region"):
    sigma_k, sigma_method = _robust_mag_scatter(group["mbar_spec"].values)
    region_sigma_info[region_name] = {
        "sigma_kwindow": sigma_k,
        "sigma_method": sigma_method,
    }

def _fit_resid_proxy_mag(row):
    fit_resid_std = row.get("fit_resid_std", np.nan)
    power_reference = row.get("P_fluc", np.nan)
    if not _finite_positive(power_reference):
        power_reference = row.get("P0", np.nan)
    if _finite_positive(fit_resid_std) and _finite_positive(abs(power_reference)):
        return float((2.5 / np.log(10.0)) * fit_resid_std / abs(power_reference))
    return np.nan

def _measurement_sigma_for_row(row):
    candidates = []
    info = region_sigma_info.get(row["region"], {})
    sigma_k = info.get("sigma_kwindow", np.nan)
    if _finite_positive(sigma_k):
        candidates.append((float(sigma_k), info.get("sigma_method", "k-window scatter")))
    sigma_fit = row.get("mbar_fit_sigma", np.nan)
    if _finite_positive(sigma_fit):
        candidates.append((float(sigma_fit), "fit covariance"))
    if candidates:
        sigma_value, sigma_method = max(candidates, key=lambda item: item[0])
        if len(candidates) > 1:
            sigma_method = "max(" + ", ".join(method for _, method in candidates) + ")"
        return sigma_value, sigma_method
    sigma_proxy = _fit_resid_proxy_mag(row)
    if _finite_positive(sigma_proxy):
        return sigma_proxy, "fit_resid_std/P_fluc proxy"
    return np.nan, "sigma unavailable"

summary_rows = []
for _, row in df_sbf.sort_values(["region", "kmin", "kmax"]).iterrows():
    sigma_measurement, sigma_method = _measurement_sigma_for_row(row)
    summary_rows.append({
        "schema_version": TABLE_SCHEMA_VERSION,
        "pipeline_version": PIPELINE_VERSION,
        "target_id": str(TARGET_GALAXY),
        "program_id": str(signal_primary_header.get("PROGRAM", "")).lstrip("0") or "0",
        "observation_id": str(signal_primary_header.get("OBSERVTN", "")),
        "signal_filter": SIGNAL_FILTER,
        "color_filter": COLOR_FILTER or "NONE",
        "record_type": "sbf_summary",
        "quantity_family": "apparent_sbf_magnitude",
        "quantity_name": "mbar_spec",
        "primary_unit": "AB mag",
        "method_id": "single_annulus_qc_selection_v1",
        "measurement_method": "azimuthal_power_spectrum_psf_fit",
        "power_correction": "P_fluc=P0-Pr",
        "calibration_applied": False,
        "distance_computed": False,
        "region_id": row["region"],
        "region_role": row.get("role", "science"),
        "region_geometry": "fixed_circular_annulus",
        "ring_selection": "fixed_literature_qc_single_annulus",
        "rin_px": row.get("rin_px", np.nan),
        "rout_px": row.get("rout_px", np.nan),
        "rin_arcsec": row.get("rin_arcsec", np.nan),
        "rout_arcsec": row.get("rout_arcsec", np.nan),
        "kmin": row.get("kmin", np.nan),
        "kmax": row.get("kmax", np.nan),
        "mbar_raw": row.get("mbar_spec_raw", np.nan),
        "mbar_corrected": row.get("mbar_spec", np.nan),
        "sigma_measurement": sigma_measurement,
        "sigma_method": sigma_method,
        "P0": row.get("P0", np.nan),
        "Pr": row.get("Pr", np.nan),
        "P_fluc": row.get("P_fluc", np.nan),
        "Pr_over_P0": row.get("Pr_over_P0", np.nan),
        "usable_fraction": row.get("usable_fraction", np.nan),
        "ring_qc_status": row.get("ring_qc_status", "FAIL"),
        "ring_qc_reasons": row.get("ring_qc_reasons", "missing QC"),
        "ring_k_stability_mag": row.get("ring_k_stability_mag", np.nan),
        "selected_for_final": bool(row.get("selected_for_final", False)),
        "selection_reason": row.get("selection_reason", ""),
        "status": row.get("status", "unknown"),
        "failure_reason": row.get("failure_reason", ""),
        "psf_method": psf_method_id,
        "psf_detector_set": psf_detector_set,
        "psf_opd_corr_id": opd_corr_id,
        "psf_opd_delta_days": float(opd_signed_delta_days),
        "psf_input_count": int(psf_input_count),
        "psf_extension": psf_selected_ext,
        "psf_native_scale_rel_error": float(psf_native_scale_rel_error),
        "psf_method_limitations": psf_method_limitations,
        "psf_siaf_prd_version": LOCAL_SIAF_PRD_VERSION,
    })
df_annulus_summary = pd.DataFrame(summary_rows)
display(df_annulus_summary)

selected_rows = df_sbf[df_sbf["selected_for_final"]].copy()
recommended_sbf = None
if selected_rows.empty:
    print("[SBF-SCIENCE] no annulus passed the encoded selection checks")
else:
    selected_row = selected_rows.iloc[0]
    selected_sigma, selected_sigma_method = _measurement_sigma_for_row(selected_row)
    main_rows = df_sbf[
        np.isclose(df_sbf["kmin"].astype(float), float(FFT_K_RANGE_MAIN[0]))
        & np.isclose(df_sbf["kmax"].astype(float), float(FFT_K_RANGE_MAIN[1]))
    ]

    def _main_value(region_name, column):
        match = main_rows[main_rows["region"].eq(region_name)]
        if match.empty:
            return np.nan
        value = match.iloc[0].get(column, np.nan)
        return float(value) if np.isfinite(value) else np.nan

    inner_mbar = _main_value("circular_inner_lit", "mbar_spec")
    outer_mbar = _main_value("circular_outer_lit", "mbar_spec")
    inner_raw = _main_value("circular_inner_lit", "mbar_spec_raw")
    outer_raw = _main_value("circular_outer_lit", "mbar_spec_raw")
    annulus_scatter = (
        float(abs(inner_mbar - outer_mbar) / 2.0)
        if np.isfinite(inner_mbar) and np.isfinite(outer_mbar)
        else np.nan
    )
    recommended_sbf = {
        "primary_quantity": "apparent_sbf_magnitude",
        "primary_unit": "AB mag",
        "measurement_method": "azimuthal_power_spectrum_psf_fit",
        "method_id": "single_annulus_qc_selection_v1",
        "selection_method": selected_sbf_selection_method,
        "selected_region": selected_row["region"],
        "selected_region_qc_status": selected_row["ring_qc_status"],
        "selected_region_qc_reasons": selected_row["ring_qc_reasons"],
        "selected_region_k_stability_mag": float(selected_row["ring_k_stability_mag"]),
        "kmin": float(selected_row["kmin"]),
        "kmax": float(selected_row["kmax"]),
        "mbar_selected_raw": float(selected_row["mbar_spec_raw"]),
        "mbar_selected": float(selected_row["mbar_spec"]),
        "sigma_selected": float(selected_sigma),
        "sigma_method": selected_sigma_method,
        "Pr_selected": float(selected_row.get("Pr", np.nan)),
        "Pr_over_P0_selected": float(selected_row.get("Pr_over_P0", np.nan)),
        "mbar_inner_raw": inner_raw,
        "mbar_inner": inner_mbar,
        "mbar_outer_raw": outer_raw,
        "mbar_outer": outer_mbar,
        "mbar_weighted_raw": float(selected_row["mbar_spec_raw"]),
        "mbar_weighted": float(selected_row["mbar_spec"]),
        "sigma_weighted_formal": float(selected_sigma),
        "annulus_scatter": annulus_scatter,
        "sigma_adopted": float(selected_sigma),
        "uses_two_annuli": False,
        "is_main_window": True,
        "calibration_applied": False,
        "distance_computed": False,
        "psf_method": psf_method_id,
        "psf_detector_set": psf_detector_set,
        "psf_opd_corr_id": opd_corr_id,
        "psf_opd_delta_days": float(opd_signed_delta_days),
        "psf_input_count": int(psf_input_count),
        "psf_extension": psf_selected_ext,
        "psf_native_scale_rel_error": float(psf_native_scale_rel_error),
        "psf_method_limitations": psf_method_limitations,
        "psf_siaf_prd_version": LOCAL_SIAF_PRD_VERSION,
        "notes": "compatibility fields named mbar_weighted contain the selected single-annulus value; no annulus weighted mean was used",
    }
    print(
        f"[SBF-SCIENCE] selected {recommended_sbf['selected_region']} at "
        f"k={recommended_sbf['kmin']:.3f}..{recommended_sbf['kmax']:.3f}"
    )
    print(
        f"[SBF-SCIENCE] apparent corrected mbar={recommended_sbf['mbar_selected']:.4f} "
        f"+/- {recommended_sbf['sigma_selected']:.4f} AB mag; not a distance"
    )
    print(
        f"[SBF-SCIENCE] QC={recommended_sbf['selected_region_qc_status']}; "
        f"{recommended_sbf['selected_region_qc_reasons']}"
    )

pipeline_variants = pd.DataFrame([
    {
        "variant": "selected_current_residual",
        "record_type": "pipeline_variant",
        "quantity_family": "apparent_sbf_magnitude",
        "quantity_name": "mbar_spec",
        "primary_unit": "AB mag",
        "method_id": "single_annulus_qc_selection_v1",
        "status": "available" if recommended_sbf is not None else "unavailable",
        "mbar_spec": recommended_sbf["mbar_selected"] if recommended_sbf is not None else np.nan,
        "selected_region": recommended_sbf["selected_region"] if recommended_sbf is not None else "",
        "calibration_applied": False,
        "distance_computed": False,
        "notes": "sigma-capped data-model residual; compact-source correction P_fluc=P0-Pr",
    },
    {
        "variant": "alternative_residual_cleaning",
        "record_type": "pipeline_variant",
        "quantity_family": "apparent_sbf_magnitude",
        "quantity_name": "mbar_spec",
        "primary_unit": "AB mag",
        "method_id": "not_run",
        "status": "not_run",
        "mbar_spec": np.nan,
        "selected_region": "",
        "calibration_applied": False,
        "distance_computed": False,
        "notes": "reserved for an independent residual-cleaning systematic check",
    },
])
display(pipeline_variants)
save_stage_table("12_summary", "annulus_summary", df_annulus_summary)
save_stage_table("12_summary", "pipeline_variants", pipeline_variants)


## Цвет в тех же кольцах

Измеряем разность `COLOR_FILTER − SIGNAL_FILTER` в тех же геометрических кольцах и с той же маской компактных объектов. Цвет сохраняется как отдельный калибровочный вход; сам ноутбук пока не переводит SBF в расстояние.


In [ ]:
print("computing color on the exact usable pixels of each fixed SBF annulus...")

required_annuli = ["circular_inner_lit", "circular_outer_lit"]
region_lookup = {reg["region"]: reg for reg in regions}
color_name = f"{COLOR_FILTER}-{SIGNAL_FILTER}" if COLOR_FILTER is not None else None
color_rows = []

def sample_color_at_signal_pixels(signal_y, signal_x):
    if color_grid_aligned:
        values = color_photometry_image[signal_y, signal_x]
        valid = color_valid[signal_y, signal_x] & np.isfinite(values)
        return values, valid

    values = np.full(signal_x.size, np.nan, dtype=float)
    valid = np.zeros(signal_x.size, dtype=bool)
    color_valid_numeric = color_valid.astype(np.uint8, copy=False)
    color_ny, color_nx = color_photometry_image.shape
    for start in range(0, signal_x.size, COLOR_WCS_SAMPLE_CHUNK):
        stop = min(start + COLOR_WCS_SAMPLE_CHUNK, signal_x.size)
        x_chunk = signal_x[start:stop].astype(float, copy=False)
        y_chunk = signal_y[start:stop].astype(float, copy=False)
        sky = signal_wcs.pixel_to_world(x_chunk, y_chunk)
        color_x, color_y = color_wcs.world_to_pixel(sky)
        inside = (
            np.isfinite(color_x) & np.isfinite(color_y)
            & (color_x >= 0.0) & (color_x <= color_nx - 1.0)
            & (color_y >= 0.0) & (color_y <= color_ny - 1.0)
        )
        if not np.any(inside):
            continue
        local_indices = np.flatnonzero(inside)
        coords = np.vstack([color_y[inside], color_x[inside]])
        sampled = ndimage.map_coordinates(
            color_photometry_image, coords, order=1, mode="constant", cval=np.nan, prefilter=False
        )
        sampled_valid = ndimage.map_coordinates(
            color_valid_numeric, coords, order=0, mode="constant", cval=0, prefilter=False
        ) > 0
        output_indices = start + local_indices
        values[output_indices] = sampled
        valid[output_indices] = sampled_valid & np.isfinite(sampled)
    return values, valid

if color_photometry_image is None:
    print("[ЦВЕТ В КОЛЬЦАХ] color image unavailable; matched-annulus colors skipped")
else:
    for region_name in required_annuli:
        reg = region_lookup[region_name]
        # Берём сохранённое множество пикселей, реально допущенное к SBF.
        # Цвет затем использует его пересечение с валидным цветовым кадром.
        sbf_pixel_mask = sbf_usable_mask_by_region[region_name]
        sbf_base_pixels = int(sbf_pixel_mask.sum())
        signal_y, signal_x = np.nonzero(sbf_pixel_mask)
        signal_values = img[signal_y, signal_x]
        color_values, sampled_valid = sample_color_at_signal_pixels(signal_y, signal_x)
        usable = sampled_valid & signal_valid[signal_y, signal_x] & np.isfinite(signal_values)
        signal_values = signal_values[usable]
        color_values = color_values[usable]

        n_raw = int(signal_values.size)
        color_overlap_fraction = (
            float(n_raw / sbf_base_pixels) if sbf_base_pixels > 0 else np.nan
        )
        row = {
            "record_type": "annulus_color",
            "method_id": "color_valid_overlap_of_sbf_window_median_ratio_v1",
            "method_variant": "joint_sigma_clip_then_ratio_of_medians",
            "quantity_family": "color_index",
            "quantity_name": "color_index",
            "primary_unit": "AB mag",
            "region_id": region_name,
            "region": region_name,
            "region_geometry": "fixed_circular_annulus",
            "ring_selection": "same_region_as_sbf",
            "rin_arcsec": reg["rin_arcsec"],
            "rout_arcsec": reg["rout_arcsec"],
            "sbf_base_pixels": sbf_base_pixels,
            "color_overlap_pixels": n_raw,
            "color_overlap_fraction": color_overlap_fraction,
            "n_raw": n_raw,
            "n_clip": 0,
            "signal_filter": SIGNAL_FILTER,
            "color_filter": COLOR_FILTER,
            "color_name": color_name,
            "sampling_mode": color_sampling_mode,
            "pixel_support": "color-valid overlap subset of the exact usable SBF spatial window",
            "signal_median": np.nan,
            "color_median": np.nan,
            "color_index": np.nan,
            "color_scatter": np.nan,
            "color_sem_proxy": np.nan,
            "ring_qc_status": ring_qc.get(region_name, {}).get("status", "FAIL"),
            "ring_qc_reasons": ring_qc.get(region_name, {}).get("reasons", "missing SBF QC"),
            "selected_for_final": bool(region_name == selected_sbf_region),
            "selection_reason": selected_sbf_selection_method if region_name == selected_sbf_region else "diagnostic ring",
            "calibration_applied": False,
            "distance_computed": False,
            "notes": "",
        }

        if n_raw <= MIN_COLOR_PIXELS:
            row["selected_for_final"] = False
            row["notes"] = "too few usable matched pixels"
            color_rows.append(row)
            continue

        _, signal_median_clip, signal_std_clip = sigma_clipped_stats(
            signal_values, sigma=COLOR_CLIP_SIGMA, maxiters=COLOR_CLIP_MAXIT
        )
        _, color_median_clip, color_std_clip = sigma_clipped_stats(
            color_values, sigma=COLOR_CLIP_SIGMA, maxiters=COLOR_CLIP_MAXIT
        )
        keep = (
            (signal_values >= signal_median_clip - COLOR_CLIP_SIGMA * signal_std_clip)
            & (signal_values <= signal_median_clip + COLOR_CLIP_SIGMA * signal_std_clip)
            & (color_values >= color_median_clip - COLOR_CLIP_SIGMA * color_std_clip)
            & (color_values <= color_median_clip + COLOR_CLIP_SIGMA * color_std_clip)
        )
        signal_robust = signal_values[keep]
        color_robust = color_values[keep]
        positive = (signal_robust > 0.0) & (color_robust > 0.0)
        signal_robust = signal_robust[positive]
        color_robust = color_robust[positive]
        n_clip = int(signal_robust.size)
        row["n_clip"] = n_clip
        if n_clip <= MIN_COLOR_PIXELS:
            row["selected_for_final"] = False
            row["notes"] = "too few positive sigma-clipped matched pixels"
            color_rows.append(row)
            continue

        signal_median = float(np.nanmedian(signal_robust))
        color_median = float(np.nanmedian(color_robust))
        color_index = float(-2.5 * np.log10(color_median / signal_median))
        pixel_colors = -2.5 * np.log10(color_robust / signal_robust)
        pixel_color_median = float(np.nanmedian(pixel_colors))
        color_mad = float(1.4826 * np.nanmedian(np.abs(pixel_colors - pixel_color_median)))
        color_sem_proxy = float(color_mad / np.sqrt(n_clip)) if np.isfinite(color_mad) else np.nan
        row.update({
            "signal_median": signal_median,
            "color_median": color_median,
            "color_index": color_index,
            "color_scatter": color_mad,
            "color_sem_proxy": color_sem_proxy,
            "notes": (
                f"same usable signal pixels as SBF; {color_sampling_mode} color sampling; "
                "scatter is a pixel-color diagnostic, not a full calibration error"
            ),
        })
        color_rows.append(row)

df_color_annuli = pd.DataFrame(color_rows)
display(df_color_annuli)

color_summary_rows = []
selected_color = None
if not df_color_annuli.empty and selected_sbf_region is not None:
    selected_match = df_color_annuli[
        df_color_annuli["region"].eq(selected_sbf_region)
        & np.isfinite(df_color_annuli["color_index"])
        & df_color_annuli["selected_for_final"].astype(bool)
    ]
    if not selected_match.empty:
        selected_color_row = selected_match.iloc[0]
        selected_color = float(selected_color_row["color_index"])
        color_summary_rows.append({
            "summary": "selected SBF-annulus color",
            "record_type": "selected_color",
            "method_id": "selected_sbf_annulus_color_v1",
            "quantity_family": "color_index",
            "quantity_name": "color_index",
            "primary_unit": "AB mag",
            "selected_region": selected_sbf_region,
            "selected_for_final": True,
            "selection_reason": selected_sbf_selection_method,
            "signal_filter": SIGNAL_FILTER,
            "color_filter": COLOR_FILTER,
            "color_name": color_name,
            "sampling_mode": color_sampling_mode,
            "color_index": selected_color,
            "sigma_proxy": float(selected_color_row.get("color_scatter", np.nan)),
            "calibration_applied": False,
            "distance_computed": False,
            "notes": "single selected SBF annulus; no averaging between annuli",
        })

available_color = (
    df_color_annuli[np.isfinite(df_color_annuli["color_index"])].copy()
    if not df_color_annuli.empty and "color_index" in df_color_annuli
    else pd.DataFrame()
)
if not available_color.empty:
    color_summary_rows.append({
        "summary": "diagnostic mean of available annulus colors",
        "record_type": "diagnostic_color",
        "method_id": "unweighted_annulus_mean_diagnostic_v1",
        "quantity_family": "color_index",
        "quantity_name": "color_index",
        "primary_unit": "AB mag",
        "selected_region": "",
        "selected_for_final": False,
        "selection_reason": "diagnostic only",
        "signal_filter": SIGNAL_FILTER,
        "color_filter": COLOR_FILTER,
        "color_name": color_name,
        "sampling_mode": color_sampling_mode,
        "color_index": float(np.nanmean(available_color["color_index"])),
        "sigma_proxy": (
            float(np.nanstd(available_color["color_index"], ddof=1))
            if len(available_color) > 1 else np.nan
        ),
        "calibration_applied": False,
        "distance_computed": False,
        "notes": "diagnostic only; never used as the calibration color",
    })

df_color_summary = pd.DataFrame(color_summary_rows)
display(df_color_summary)
save_stage_table("13_color", "annulus_colors", df_color_annuli)
save_stage_table("13_color", "color_summary", df_color_summary)

# Каноническая длинная таблица: одна строка = одна физическая величина.
measurement_rows = []
for _, row in df_annulus_summary.iterrows():
    measurement_rows.append({
        "record_type": "measurement",
        "method_id": row.get("method_id", "single_annulus_qc_selection_v1"),
        "quantity_family": "apparent_sbf_magnitude",
        "quantity_name": "mbar_corrected",
        "value": row.get("mbar_corrected", np.nan),
        "uncertainty": row.get("sigma_measurement", np.nan),
        "unit": "AB mag",
        "region_id": row.get("region_id", ""),
        "kmin": row.get("kmin", np.nan),
        "kmax": row.get("kmax", np.nan),
        "selected_for_final": bool(row.get("selected_for_final", False)),
        "qc_status": row.get("ring_qc_status", ""),
        "qc_reasons": row.get("ring_qc_reasons", ""),
        "calibration_applied": False,
        "distance_computed": False,
    })
for _, row in df_color_annuli.iterrows():
    measurement_rows.append({
        "record_type": "measurement",
        "method_id": row.get("method_id", "color_valid_overlap_of_sbf_window_median_ratio_v1"),
        "quantity_family": "color_index",
        "quantity_name": row.get("color_name", color_name),
        "value": row.get("color_index", np.nan),
        "uncertainty": row.get("color_scatter", np.nan),
        "unit": "AB mag",
        "region_id": row.get("region", ""),
        "kmin": np.nan,
        "kmax": np.nan,
        "selected_for_final": bool(row.get("selected_for_final", False)),
        "qc_status": row.get("ring_qc_status", ""),
        "qc_reasons": row.get("ring_qc_reasons", ""),
        "calibration_applied": False,
        "distance_computed": False,
    })
df_measurements_long = pd.DataFrame(measurement_rows)
display(df_measurements_long)
save_stage_table("14_measurements", "long", df_measurements_long)
